# FlashRAG 全流程实战向导

**FlashRAG** 是由中国人民大学 NLP 团队开源的 RAG（检索增强生成）研究框架。
本 Notebook 基于 **官方仓库 `main` 分支**（https://github.com/RUC-NLPIR/FlashRAG）
的实际源码整理，所有 API 均已与官方代码对照验证。

| 章节 | 内容 |
|------|------|
| 一 | 安装与环境配置 |
| 二 | 架构概览 |
| 三 | 配置系统（Config / YAML） |
| 四 | 知识库构建（分块 → JSONL → 构建 FAISS 索引） |
| 五 | 检索器（Dense / BM25 / 多路混合） |
| 六 | 重排序器（CrossReranker） |
| 七 | 生成器（OpenAI API / 本地 HF / PromptTemplate） |
| 八 | Pipeline 实战（Naive / FLARE / 自定义） |
| 九 | 结果评估 |
| **十** | **完整端到端流程（一键跑通）** |
| 十一 | 总结 |

## 一、安装与环境配置

In [ ]:
# ============================================================
# 安装 FlashRAG 及核心依赖
# ------------------------------------------------------------
# flashrag-dev          : FlashRAG 框架本体（PyPI 官方包名为 flashrag-dev，
#                         加 --pre 以安装最新预发布版）
# transformers          : HuggingFace 模型加载（检索/生成均依赖）
# faiss-cpu             : FAISS 向量索引库 CPU 版
#                         ⚠ 官方推荐用 conda 安装以避免兼容问题：
#                           conda install -c pytorch faiss-cpu=1.8.0
#                         GPU 版：conda install -c pytorch -c nvidia faiss-gpu=1.8.0
# bm25s                 : BM25 稀疏检索后端（FlashRAG 默认 bm25 backend）
# datasets              : HuggingFace 数据集加载工具（FlashRAG 内部使用）
# openai                : OpenAI Python SDK（兼容所有 /v1/chat/completions 端点）
# tiktoken              : OpenAI tokenizer（OpenaiGenerator 内部依赖）
# langchain-text-splitters : LangChain 文档分块工具（FlashRAG 无内置分块）
# ============================================================
%pip install flashrag-dev --pre transformers bm25s datasets openai tiktoken langchain-text-splitters --quiet  # 安装所有核心依赖，--quiet 减少冗余输出

# ---- faiss 建议用 conda 安装（pip 版本在部分系统存在兼容问题）----
# -c pytorch : 指定从 pytorch conda 频道下载，并非安装 PyTorch 本身。
#              FAISS 由 Meta (Facebook AI Research) 开发，与 PyTorch 同属一个团队，
#              因此 FAISS 的官方 conda 包发布在 pytorch 频道（anaconda.org/pytorch/faiss-cpu）
# !conda install -c pytorch faiss-cpu=1.8.0 -y

In [ ]:
# ============================================================
# 验证关键依赖是否安装成功，打印各库版本号
# ============================================================
import importlib  # 动态导入模块工具，用于版本检查
import sys        # 系统信息模块，用于打印 Python 解释器版本

# 待检查的库列表，List[Tuple[str, str]] 类型
libs = [  # List[Tuple[str, str]]: 待检查的库列表，每项为 (import_name, display_name)
    ("flashrag",    "FlashRAG"),                          # (str, str): FlashRAG 框架
    ("transformers","Transformers"),                       # (str, str): HuggingFace Transformers
    ("faiss",       "FAISS"),                              # (str, str): 向量索引库
    ("bm25s",       "bm25s"),                              # (str, str): BM25 稀疏检索后端
    ("datasets",    "HuggingFace Datasets"),               # (str, str): 数据集加载工具
    ("openai",      "OpenAI SDK"),                         # (str, str): OpenAI Python SDK
    ("tiktoken",    "tiktoken"),                           # (str, str): OpenAI tokenizer
    ("langchain_text_splitters", "LangChain TextSplitters"),  # (str, str): 文本分块工具
]

print(f"Python 版本: {sys.version}")  # 打印 Python 解释器版本字符串
print("-" * 40)                         # 打印分隔线，长度固定 40
# 遍历 libs 列表，每项包含 (import_name, display_name) 两个字符串
for import_name, display_name in libs:  # import_name: str 模块名，display_name: str 展示名
    try:  # 尝试动态导入模块，失败时进入 except 分支
        # importlib.import_module(name) : 动态导入模块，返回 module 对象
        mod = importlib.import_module(import_name)  # module 对象：成功导入的库
        # getattr(obj, attr, default) : 安全获取属性，缺少 __version__ 时返回 "已安装"
        ver = getattr(mod, "__version__", "已安装")  # str: 版本号字符串
        print(f"✓ {display_name:<25} {ver}")  # 打印安装成功的库及其版本
    except ImportError:  # 模块不存在时抛出，提示用户重新安装
        print(f"✗ {display_name:<25} 未安装，请重新运行上方安装命令")  # 打印未安装提示

## 二、FlashRAG 架构概览

FlashRAG 整体数据流（按源码实际执行路径）：

```
原始文档
   ↓ ① RecursiveCharacterTextSplitter (LangChain)  文档分块
JSONL 语料（{id, contents, title}）
   ↓ ② Index_Builder.build_index()     flashrag.retriever.index_builder.Index_Builder
索引文件（Dense: {method}_{type}.index / BM25: {save_dir}/bm25/ 目录）
   ↓
用户问题 → ③ retriever.batch_search(queries)    批量检索（实际方法名）
                ↓
            ④ reranker.rerank(queries, docs)     CrossReranker（实际类名）
               返回 (reranked_docs, scores) 元组
                ↓
            ⑤ prompt_template.get_string(       PromptTemplate（实际方法名）
                   question=q, retrieval_result=r)
                ↓
            ⑥ generator.generate(prompts)        OpenaiGenerator / HFCausalLMGenerator
                ↓
            ⑦ evaluator.evaluate(dataset)        Evaluator
```

**步骤 ③~⑦ 均由 Pipeline 一键封装**（见第八章）。

### 组件一览（均经官方源码验证）

| 类别 | 官方模块路径 | 类/函数名 |
|------|------------|----------|
| 检索器 | `flashrag.retriever` | `DenseRetriever` / `BM25Retriever` |
| 多路检索 | Config: `use_multi_retriever=True` | `MultiRetrieverRouter`（自动选择） |
| 重排序 | `flashrag.retriever` | `CrossReranker` / `BiReranker` |
| 索引构建 | `flashrag.retriever.index_builder` | `Index_Builder` |
| 生成器(API) | `flashrag.generator` | `OpenaiGenerator` |
| 生成器(本地) | `flashrag.generator` | `HFCausalLMGenerator` / `VLLMGenerator` |
| Prompt 模板 | `flashrag.prompt` | `PromptTemplate` |
| Pipeline | `flashrag.pipeline` | `SequentialPipeline` / `FLAREPipeline` / `SelfRAGPipeline` 等 |
| 评估器 | `flashrag.evaluator` | `Evaluator` |
| 配置 | `flashrag.config` | `Config`（YAML 驱动 + `basic_config.yaml` 默认值） |

## 三、配置系统

### 3.1 创建 YAML 配置文件

FlashRAG 内置 `basic_config.yaml` 提供所有参数的默认值，
自定义 YAML 只需覆盖与默认值不同的字段即可。

**重要路径规则**（来自 `config.py` 源码）：
```
dataset_path = os.path.join(data_dir, dataset_name)
```
数据集文件应放在 `{data_dir}/{dataset_name}/test.jsonl`。

In [ ]:
import os    # 操作系统接口，用于创建目录和读取环境变量
import yaml  # YAML 序列化/反序列化库，用于写入配置文件

# ============================================================
# 定义工作目录结构
# ============================================================
WORKSPACE    = "flashrag_workspace"          # str: 根工作目录
CORPUS_DIR   = f"{WORKSPACE}/corpus"        # str: 语料存放目录
INDEX_DIR    = f"{WORKSPACE}/index"         # str: 检索索引存放目录
OUTPUT_DIR   = f"{WORKSPACE}/output"        # str: 实验结果存放目录
DATASET_NAME = "test_data"                   # str: 数据集名称（影响 dataset_path 计算）
# FlashRAG Config 内部会计算：dataset_path = data_dir / dataset_name
# 所以数据集文件须放在 WORKSPACE/DATASET_NAME/test.jsonl
#
# ---- 这里【数据集】的含义 ----
# 指用于测试 RAG Pipeline 效果的【问答评估集】，是一个 JSONL 文件（test.jsonl）。
# 该文件与模型训练数据无关，每行一条 JSON，包含以下字段：
#   question       : str       — 问题文本
#   golden_answers : List[str] — 标准参考答案（用于 EM/F1/Acc 等指标计算）
#   id             : str       — 问题唯一标识（可选）
# 示例：
#   {"id": "q0", "question": "RAG 的三个发展阶段？",
#    "golden_answers": ["朴素 RAG、高级 RAG、模块化 RAG"]}
# Pipeline.run(dataset) 读取该文件后，对每条 question 执行「检索 → 生成」，
# 再将 pred 与 golden_answers 对比，计算 EM/F1/retrieval_recall 等评估指标。
DATA_SUBDIR  = f"{WORKSPACE}/{DATASET_NAME}"  # str: 实际数据集目录（test.jsonl 须放在此处）

# os.makedirs(path, exist_ok=True) : 递归创建多级目录，exist_ok=True 时已存在不报错
for d in [CORPUS_DIR, INDEX_DIR, OUTPUT_DIR, DATA_SUBDIR]:  # 遍历所有需要创建的目录
    os.makedirs(d, exist_ok=True)  # 逐一创建目录，已存在时跳过

# os.environ.get(key, default) : 读取环境变量，不存在时返回 default
# 设置环境变量（Windows）: set DEEPSEEK_API_KEY=sk-xxxx
# 设置环境变量（Linux/Mac）: export DEEPSEEK_API_KEY=sk-xxxx
API_KEY  = os.environ.get("DEEPSEEK_API_KEY", "your-api-key-here")  # str: API 密钥
BASE_URL = "https://api.deepseek.com/v1"   # str: API 根地址（可替换为中转站）

# ============================================================
# 自定义配置字典，仅覆盖与 basic_config.yaml 默认值不同的字段
# 未覆盖的字段由 FlashRAG 内置默认值自动填充
# ============================================================
config_dict = {  # Dict: FlashRAG 配置字典，仅覆盖与 basic_config.yaml 默认值不同的字段
    # ---- 全局设置 ----
    "save_dir":      OUTPUT_DIR,   # str: 评估结果保存目录（Config 内部会自动追加时间戳子目录）
    "gpu_id":        "0",          # str: 使用的 GPU 编号；"" 或不设置表示仅 CPU
    "seed":          2024,          # int: 随机种子，保证实验可复现
    "save_note":     "demo",        # str: 保存目录的后缀标识

    # ---- 语料与索引路径 ----
    # corpus_path : JSONL 格式，每行 {"id": "0", "contents": "文本", "title": "..."}
    "corpus_path":   f"{CORPUS_DIR}/my_corpus.jsonl",   # str: 语料文件路径
    # index_path  : DenseRetriever 加载时要求索引文件必须已存在（需先用 Index_Builder 构建）
    # 文件名格式（来自源码）：{retrieval_method}_{faiss_type}.index（如 bge_Flat.index）
    "index_path":    f"{INDEX_DIR}/bge_Flat.index",       # str: FAISS 索引文件路径（占位，后续更新）

    # ---- 检索器设置 ----
    # retrieval_method : 检索方式名称，同时作为模型别名
    #   常用值："e5"（intfloat/e5-base-v2）、"bge"（BAAI/bge-base-en 或 bge-small-zh-v1.5 等）、"bm25"
    # ---- 常用检索器模型参考 ----
    #   稠密检索（Dense）— 需编码模型，支持语义检索：
    #     中文（窗口均为 512 token，bge-m3 为 8192）：
    #       BAAI/bge-small-zh-v1.5   轻量（~24M），窗口 512，速度快，当前使用，适合资源受限场景
    #       BAAI/bge-base-zh-v1.5    中等（~102M），窗口 512，精度更高
    #       BAAI/bge-large-zh-v1.5   较大（~326M），窗口 512，精度最高，显存要求高
    #       BAAI/bge-m3              多语言（~570M），窗口 8192，支持稠密+稀疏+多向量三种检索
    #     英文（窗口均为 512 token）：
    #       intfloat/e5-base-v2      英文通用，均衡性能
    #       intfloat/e5-large-v2     英文，精度更高
    #       BAAI/bge-base-en-v1.5    英文，MTEB 榜单主流
    #   稀疏检索（BM25）— 无需模型，关键词匹配，速度极快：
    #     retrieval_method="bm25"，无需 retrieval_model_path，无上下文窗口限制
    #     对专业术语/缩写匹配效果好，对同义词召回弱于稠密检索
    #   稠密与稀疏不互斥，可同时启用（混合检索）：
    #     Config 中设置 use_multi_retriever=True，并配置 retriever_list
    #     两路各自检索后用 RRF 算法融合排名，兼顾语义理解和关键词精确匹配
    #     详见第 5.3 节（MultiRetrieverRouter）
    #   选型建议：
    #     中文场景               → bge-small-zh-v1.5（轻量）或 bge-base-zh-v1.5（精度）
    #     chunk 文本较长（>100字）→ 将 retrieval_query_max_length 调至模型窗口上限
    #     需要多语言/长文档      → bge-m3（窗口 8192）
    #     资源极度受限/纯关键词  → bm25
    #     最优效果（两者互补）   → 混合检索（见第5.3节）
    "retrieval_method":     "bge",                        # str: 检索方式名称（与 retrieval_model_path 中使用的 BGE 系列模型保持一致）
    # retrieval_model_path : 编码模型路径（本地目录或 HuggingFace Hub ID），默认值为 ~（None）
    # 若不指定（保持 None），Config 以 retrieval_method 的值为 key 在 model2path 中查路径；
    #   basic_config.yaml 中 model2path 的实际内容（仅以下几项）：
    #     e5         → intfloat/e5-base-v2
    #     bge        → BAAI/bge-base-en-v1.5   ← 注意：英文模型！
    #     contriever → facebook/contriever
    #     llama2-*   → 对应 LLaMA-2 HuggingFace 路径
    #   ⚠ 本 Notebook 使用中文模型 bge-small-zh-v1.5，不在默认映射表中，
    #     必须显式设置 retrieval_model_path，否则会错误加载英文 bge-base-en-v1.5
    #   若已设置完整路径（如 HuggingFace Hub ID 或本地绝对路径），则直接使用，不查表
    "retrieval_model_path": "BAAI/bge-small-zh-v1.5",   # str: 中文轻量模型（~24M）
    "retrieval_topk":       5,        # int: 检索返回的 Top-K 文档数
    # retrieval_query_max_length : 编码模型单次处理文本的最大 token 数（索引构建和检索两阶段均生效）
    #   basic_config.yaml 默认值为 128（约 100~120 汉字），未显式设置时自动使用该默认值
    #   ⚠ 当前 chunk_size=200，默认 128 会截断语料文本，导致索引只编码了前半段，降低检索质量
    #   应与所用模型的上下文窗口上限对齐：bge-small/base/large-zh-v1.5 窗口均为 512
    "retrieval_query_max_length": 512,  # int: 编码最大 token 数（对齐 bge-small-zh-v1.5 窗口上限 512）
    # retrieval_batch_size : 构建 FAISS 索引时每批编码的文档数量
    #   仅在 Index_Builder.build_index() 阶段生效，索引构建完成后不再使用
    #   值越大 GPU 利用率越高、构建越快，但显存占用也越大
    #   建议值：GPU ≥ 8GB → 512；GPU 4-8GB → 256；仅 CPU 或显存不足 → 64~128
    "retrieval_batch_size": 256,       # int: 每批编码文档数（GPU 4-8GB 下推荐值）
    # retrieval_use_fp16 : 编码时是否使用半精度浮点（FP16）
    #   True  → 显存减半、速度约快 1.5~2 倍，精度损失可忽略（需 RTX 20 系及以上 GPU）
    #   False → FP32 全精度，仅 CPU 或旧 GPU 时使用（basic_config.yaml 默认值）
    # 注意：faiss_type（Flat/IVF/HNSW）不是 Config 字段，无法在 YAML 中设置，
    #   只能在调用 Index_Builder 时直接传参（见第四章索引构建代码）
    "retrieval_use_fp16":   True,        # bool: 使用 FP16 加速（有现代 GPU 时设为 True）
    # bm25_backend : BM25 后端库；官方有效值为 "bm25s"（默认）或 "pyserini"（需 Java）
    # 注意："rank_bm25" 不是 FlashRAG 支持的 backend，会导致运行错误
    "bm25_backend":         "bm25s",  # str: BM25 后端，推荐 "bm25s"

    # ---- 重排序器设置（仅在 use_reranker=True 时生效）----
    "use_reranker":         False,    # bool: 是否启用重排序（Pipeline 运行时读取）
    # rerank_model_name : 重排序模型的标识名称（相当于给模型起个别名）
    #   当 rerank_model_path 未指定时，Config 才会用此名在 model2path 中自动查找路径
    #   已显式设置 rerank_model_path 时，此字段仅作标识用，不触发自动查找
    # ---- 常用 Reranker 模型参考 ----
    #   中文/中英双语（本地）：
    #     BAAI/bge-reranker-base        轻量（~280M），窗口 512，中英双语，当前使用
    #     BAAI/bge-reranker-large       效果更好，窗口 512，显存要求更高
    #     BAAI/bge-reranker-v2-m3       多语言，窗口 8192，长文档友好
    #     BAAI/bge-reranker-v2-gemma    多语言，窗口 8192，效果最强，资源消耗大
    #     maidalun1020/bce-reranker-base_v1  中英双语，BCE 框架
    #   英文（本地）：
    #     cross-encoder/ms-marco-MiniLM-L-6-v2  轻量经典，纯英文
    #   长文档（本地）：
    #     Alibaba-NLP/gte-reranker-modernbert-8b  窗口长，效果强
    #   API 调用（无需本地部署）：
    #     Cohere rerank-multilingual-v3.0 / rerank-english-v3.0
    # 选型建议：中文场景首选 bge-reranker-base/large；长 chunk（>300字）用 v2-m3
    "rerank_model_name":    "bge-reranker-base",  # str: reranker 模型标识名
    # rerank_model_path : 优先级高于 rerank_model_name；直接指定模型路径或 HuggingFace Hub ID
    "rerank_model_path":    "BAAI/bge-reranker-base",  # str: reranker 模型路径
    "rerank_topk":          3,         # int: 重排序后保留的文档数
    # rerank_max_length : CrossReranker 对每条文档单独与 query 拼接后送入模型的最大 token 数
    #   每个 (query, doc) 对独立处理，共处理 retrieval_topk 对，按 rerank_batch_size 分批并行
    #   （注意：不是将 query 与所有 topk 文档一次性拼接，而是每对单独过模型）
    #   超出部分会被截断；值越大保留内容越完整，但推理越慢、显存占用越大
    #   bge-reranker-base 上下文窗口为 512，此处设为满窗口；
    #   若换用 bge-reranker-v2-m3（窗口 8192）可适当调大
    "rerank_max_length":    512,        # int: 单条 (query+doc) 拼接文本的最大 token 数（bge-reranker-base 满窗口）
    # rerank_batch_size : 每次并行处理的 (query, document) 对数量
    #   值越大吞吐越高，但显存占用越大；显存不足时调低至 16 或 32
    "rerank_batch_size":    64,         # int: 每批并行打分的文档对数

    # ---- 生成器设置（OpenAI 兼容接口）----
    # framework 有效值（来自 get_generator 源码）："openai" / "hf" / "vllm" / "fschat"
    "framework":            "openai",                     # str: 生成器后端类型
    # generator_model : API 模型 ID（framework="openai" 时）或本地模型路径/别名（hf/vllm 时）
    "generator_model":      "deepseek-chat",              # str: 生成模型 ID
    "generator_max_input_len": 4096,   # int: Prompt 最大 token 数（超出自动截断）
    "generator_batch_size": 1,          # int: 生成批次大小（openai 框架下每次并发请求数）
    "generation_params": {              # Dict: 生成超参数，传给 API 的请求体
        "max_tokens":   512,            # int: 生成回答的最大 token 数
        "temperature":  0.1,            # float: 采样温度，越低越确定
    },                                  
    # openai_setting : 直接传给 AsyncOpenAI(**openai_setting) 的参数（来自源码）
    "openai_setting": {                 # Dict: OpenAI SDK 初始化参数
        "api_key":  API_KEY,            # str: API 密钥
        "base_url": BASE_URL,           # str: API 根地址
    },                                  # end of openai_setting

    # ---- 数据集设置 ----
    # Config 源码：dataset_path = os.path.join(data_dir, dataset_name)
    # 因此数据集文件应位于：{data_dir}/{dataset_name}/test.jsonl
    "data_dir":             WORKSPACE,        # str: 数据集根目录
    "dataset_name":         DATASET_NAME,     # str: 数据集名称（作为子目录名）
    "split":                ["test"],         # List[str]: 使用的数据集分片
    "test_sample_num":      None,             # int|None: 评估采样数量，None 表示全量

    # ---- 评估指标设置 ----
    # 指标名称来自 flashrag/evaluator/metrics.py 各类的 metric_name 属性
    # 注意：FlashRAG 使用 "em"、"f1"、"acc"，而非 RAGAS 的 "answer_em"、"answer_f1"
    # 注意：FlashRAG 使用 "retrieval_recall"，而非 "retrieval_hit@k" 或 "retrieval_mrr"
    "metrics": ["em", "f1", "acc", "retrieval_recall"],  # List[str]: 启用的评估指标
    "save_metric_score":      True,   # bool: 是否将指标结果写入 metric_score.txt
    "save_intermediate_data": False,  # bool: 是否保存每条样本的详细得分
    # metric_setting : 部分指标的额外参数字典
    #   retrieval_recall_topk : int，检索召回率在前 K 篇里计算
    #     应与 retrieval_topk 保持一致，而非 rerank_topk：
    #     retrieval_recall 评估的是检索器（重排序之前）的原始输出能否召回正确文档，
    #     检索器返回几篇就在几篇里评估，设为 rerank_topk 会人为压低召回率
    "metric_setting": {               # Dict: 指标额外参数
        "retrieval_recall_topk": 5,  # int: 应与 retrieval_topk=5 一致，而非 rerank_topk=3
    },                                
}  

# yaml.dump : 将 Python 字典序列化为 YAML 格式并写入文件
config_path = f"{WORKSPACE}/config.yaml"  # str: 配置文件路径
with open(config_path, "w", encoding="utf-8") as f:  # 打开文件，utf-8 编码写入
    # allow_unicode=True 保留中文字符，default_flow_style=False 使用块式格式
    yaml.dump(config_dict, f, allow_unicode=True, default_flow_style=False, sort_keys=False)  # 序列化写入 YAML

print(f"配置文件已写入: {config_path}")              # 打印配置文件保存路径
print(f"数据集目录   : {DATA_SUBDIR}  (存放 test.jsonl 的实际路径)")  # 提示数据集存放位置
print(f"API Key      : {'*' * 8 + API_KEY[-4:] if len(API_KEY) > 4 else '(未设置)'}")  # 打印 API Key（后 4 位明文，其余遮蔽）

### 3.2 加载 Config 对象

In [ ]:
# flashrag.config.Config : FlashRAG 配置中心
# 内部加载 basic_config.yaml（默认值），再与用户 YAML 合并，最终合并运行时覆盖字典
# 优先级：运行时 override_dict > 用户 YAML > basic_config.yaml 默认值
from flashrag.config import Config  # FlashRAG 配置管理类，读取 YAML 并合并默认值

# Config(config_file_path, override_dict) :
#   config_file_path : str，用户 YAML 文件路径
#   override_dict    : Dict，运行时覆盖参数；传 {} 表示完全使用 YAML 中的值
# 返回值 : Config 对象
#   支持属性访问（config.retrieval_topk）和字典访问（config["retrieval_topk"]）
config = Config(config_path, {})  # Config 对象：用空 override_dict 完全依赖 YAML 中的值

print("Config 加载成功！")                                    # 打印加载成功提示
print(f"  检索方法    : {config['retrieval_method']}")           # 打印检索方式
print(f"  检索 Top-K  : {config['retrieval_topk']}")             # 打印检索返回文档数
print(f"  生成框架    : {config['framework']}")                   # 打印生成器类型
print(f"  生成模型    : {config['generator_model']}")             # 打印生成模型名称
# dataset_path 是 Config 自动派生的字段，不需要在 YAML 中手动填写
# Config.__init__() 内部执行：dataset_path = os.path.join(data_dir, dataset_name)
# 即由你填写的 data_dir="flashrag_workspace" 和 dataset_name="test_data" 自动拼接而来
# 框架用此路径去寻找 test.jsonl（完整路径：flashrag_workspace/test_data/test.jsonl）
print(f"  数据集路径  : {config['dataset_path']}")                # 打印数据集实际路径（自动派生，非手动填写）

# ---- 消融实验示例：动态覆盖 Top-K，不修改 YAML 文件 ----
# Config(path, override_dict) : override_dict 优先级最高，可在不修改 YAML 的情况下覆盖参数
config_topk3 = Config(config_path, {"retrieval_topk": 3, "rerank_topk": 2})  # Config: 覆盖 topk 参数的消融实验配置
# 打印消融实验配置，验证覆盖是否生效
print(f"\n消融实验配置：Top-K={config_topk3['retrieval_topk']}, Rerank-K={config_topk3['rerank_topk']}")  # 打印覆盖后的参数值

## 四、知识库构建

### 4.1 文档分块与语料写入

> **FlashRAG 本身没有内置文档分块工具**，其定位是 RAG 算法研究框架，
> 假设使用者已自行准备好分块后的语料。
> 本节使用 **LangChain** 的 `RecursiveCharacterTextSplitter` 进行分块，
> 这是目前对中文支持最好、功能最完整的通用分块方案。

FlashRAG 要求语料为 **JSONL 格式**，每行必须包含：
- `id`：唯一标识（str）
- `contents`：chunk 文本内容（str）

可选字段：`title` 等元数据。

In [ ]:
# ============================================================
# FlashRAG 本身没有内置文档分块工具，推荐使用 LangChain 的
# RecursiveCharacterTextSplitter 进行分块：
#   - 按优先级逐级尝试多个分隔符（段落 → 句号 → 空格 → 字符），
#     确保尽量在语义边界处切分
#   - 对中文友好，可将句号、感叹号等中文标点加入分隔符列表
#   - 支持 chunk_overlap（相邻 chunk 保留重叠），避免上下文断裂
# ============================================================
import json                                              # JSON 序列化模块
from typing import List, Dict                            # 类型注解
from langchain_text_splitters import (                   # LangChain 分块工具
    RecursiveCharacterTextSplitter,                      # 递归字符分块器（推荐）
)

# ============================================================
# 初始化 RecursiveCharacterTextSplitter
# ------------------------------------------------------------
# chunk_size    : int，每个 chunk 的最大字符数
# chunk_overlap : int，相邻 chunk 的重叠字符数，用于保留上下文
# separators    : List[str]，按优先级从高到低的分隔符列表：
#                 双换行（段落） > 单换行 > 中文句号/感叹号/问号 > 空格 > 逐字符
# length_function: Callable，计算文本长度的函数（默认 len，按字符数计算）
# is_separator_regex: bool，分隔符是否为正则表达式，False 表示普通字符串
# ============================================================
text_splitter = RecursiveCharacterTextSplitter(  # RecursiveCharacterTextSplitter: 递归字符分块器实例
    chunk_size=200,                          # int: 每块最大字符数
    # chunk_overlap : 相邻两块的重叠字符数
    #   上一块末尾的 N 个字会原样出现在下一块开头，防止一句话恰好被切断
    #   示例（chunk_size=20, chunk_overlap=10）：
    #     chunk_0："AAAAAAAAAA BBBBBBBBBB"
    #     chunk_1："BBBBBBBBBB CCCCCCCCCC"  ← BBBBBBBBBB 重复出现（即重叠部分）
    #   对比 chunk_overlap=0：
    #     chunk_0："AAAAAAAAAA BBBBBBBBBB"
    #     chunk_1："CCCCCCCCCC ..."         ← BBBBBBBBBB 这段上下文被彻底切断，
    #                                          检索时跨越该边界的问题可能无法命中
    #   代价是语料总体积略增大，但跨边界问题的召回率更高
    chunk_overlap=30,                        # int: 相邻块重叠字符数（末尾 30 字出现在下一块开头）
    # separators : 分隔符优先级列表，从左到右逐级尝试
    #   先用最高优先级分隔符切；若切完某段仍超过 chunk_size，再对该段尝试下一级分隔符，
    #   目标是尽量在语义完整的边界处切断，而非生硬地按第 N 个字截断
    #   优先级（从高到低）：
    #     "\n\n"  段落边界（最优先，保留完整段落）
    #     "\n"    换行
    #     "。"    中文句号
    #     "！"    中文感叹号
    #     "？"    中文问号
    #     "；"    中文分号
    #     " "     空格
    #     ""      逐字符兜底（最后手段，强制按字切）
    separators=["\n\n", "\n", "。", "！", "？", "；", " ", ""],  # List[str]: 分隔符优先级列表（段落→换行→中文标点→空格→字符）
    length_function=len,                     # Callable: 字符计数函数
    is_separator_regex=False,                # bool: 分隔符为普通字符串
)

# ============================================================
# 示例语料：关于 RAG 技术的中文知识片段
# ============================================================
# raw_texts : List[Dict]，分块前的原始文档，字段格式由你自定义，框架不直接读取
#   这里用 title + content（注意：是 content 单数，不是 contents）
#   raw_texts 中没有 id 字段也没有 contents 字段，这是正常的：
#     id       和 contents 是下方写入循环自动生成/重命名的，是语料 JSONL 的规定格式
#   字段对应关系（raw_texts → corpus JSONL）：
#     content  → contents : 分块后的 chunk 文本（content 是你起的名，contents 是框架要求的名）
#     title    → title    : 原样保留
#     （无）   → id      : 循环中自动生成，格式为 "{doc_idx}_{chunk_idx}"
raw_texts = [  # List[Dict]: 原始文档列表，每项含 title 和 content 两个字段
    {                                  # Dict: 第 1 篇文档
        "title": "RAG 技术综述",   # str: 文档标题，用于 JSONL 的 title 字段
        "content": (               # str: 文档正文，将被分块写入 contents 字段
            "检索增强生成（RAG）将信息检索与大语言模型结合，解决幻觉和知识更新问题。\n\n"
            "RAG 三阶段：朴素 RAG（仅检索生成）、高级 RAG（加入重排序/查询改写）、"
            "模块化 RAG（各组件可独立替换）。"
        )
    },                             # end of 第 1 篇文档
    {                              # Dict: 第 2 篇文档
        "title": "检索技术",        # str: 文档标题
        "content": (               # str: 文档正文
            "向量检索（Dense）通过编码模型将文本映射为稠密向量，利用余弦相似度或内积检索。\n\n"
            "BM25 是经典稀疏检索算法，对关键词匹配效果好，但对同义词召回弱于稠密检索。\n\n"
            "混合检索（Hybrid）用 RRF 融合两种结果，兼顾语义和关键词匹配。"
        )
    },                             # end of 第 2 篇文档
    {                              # Dict: 第 3 篇文档
        "title": "FlashRAG 框架",   # str: 文档标题
        "content": (               # str: 文档正文
            "FlashRAG 是中国人民大学开源 RAG 框架，内置 Naive RAG、Self-RAG、FLARE 等 20+ 算法，"
            "支持 DenseRetriever、BM25Retriever，"
            "以及 OpenaiGenerator、HFCausalLMGenerator 三种生成器后端。\n\n"
            "所有参数由 YAML 配置文件集中管理，通过 basic_config.yaml 提供默认值，"
            "切换算法只需修改配置，无需改动代码。"
        )
    },                             # end of 第 3 篇文档
    {                              # Dict: 第 4 篇文档
        "title": "高级 RAG 算法",   # str: 文档标题
        "content": (               # str: 文档正文
            "FLARE（Forward-Looking Active REtrieval）在生成过程中遇到低置信度 token 时"
            "主动发起新一轮检索，将检索和生成交织进行，适合多跳推理。\n\n"
            "Self-RAG 让模型生成反思 token（ISREL/ISSUP/ISUSE）"
            "自主判断是否需要检索，避免对所有问题都强制检索的低效。"
        )
    },                             # end of 第 4 篇文档
]

# ---- 分块并写入 JSONL 语料文件 ----
corpus_path = f"{CORPUS_DIR}/my_corpus.jsonl"  # str: 语料文件路径
total_chunks = 0   # int: chunk 计数器，记录写入总数

# 打开语料文件（覆盖写模式），utf-8 编码支持中文
with open(corpus_path, "w", encoding="utf-8") as f:  # 文件句柄 f，写入完毕后自动关闭
    # 遍历所有原始文档，doc_idx 为文档序号（int），doc 为 Dict
        for doc_idx, doc in enumerate(raw_texts):  # doc_idx: int 文档序号，doc: Dict（title/content）
            # text_splitter.split_text(text) : 返回 List[str]
            # 长度 = 实际切分出的 chunk 数量，取决于文本长度与 chunk_size
            chunks: List[str] = text_splitter.split_text(doc["content"])  # List[str]: 切分后的文本块列表
        # 遍历当前文档的所有 chunk，chunk_idx 为该文档内的序号
        for chunk_idx, chunk_text in enumerate(chunks):  # chunk_idx: int 块序号，chunk_text: str 块文本
            record = {                                   # Dict: 单条 JSONL 记录（符合 FlashRAG 格式）
                "id":       f"{doc_idx}_{chunk_idx}",   # str: 全局唯一 id
                "title":    doc["title"],                # str: 来源文档标题
                "contents": chunk_text,                  # str: chunk 正文（必填字段）
            }
            # json.dumps + "\n" : 每条文档占一行，符合 JSONL 规范
            f.write(json.dumps(record, ensure_ascii=False) + "\n")  # 写入一行 JSON
            total_chunks += 1  # 每写入一条 chunk，计数器加 1

print(f"语料文件: {corpus_path}")                              # 打印语料文件保存路径
print(f"原始文档: {len(raw_texts)} 篇  →  chunk 总数: {total_chunks}")  # 打印分块统计结果

### 4.2 构建 FAISS 检索索引

> **注意（来自官方源码）**：`DenseRetriever.load_index()` 在索引文件不存在时直接报错，
> **不会自动构建索引**。必须先用 `Index_Builder` 显式构建并保存索引文件，
> 再初始化 `DenseRetriever`。

In [ ]:
# flashrag.retriever.index_builder.Index_Builder :
# 负责将 JSONL 语料编码为向量并构建 FAISS 索引文件（或 BM25 索引目录）
from flashrag.retriever.index_builder import Index_Builder  # FAISS / BM25 索引构建器

# Index_Builder.__init__ 关键参数（来自源码）：
#   retrieval_method : str，检索方式名称（"e5"、"bge" 等，用于选择 encoder 类型）
#   model_path       : str，编码模型路径或 HuggingFace Hub 模型 ID
#   corpus_path      : str，JSONL 语料文件路径
#   save_dir         : str，索引文件保存目录
#   max_length       : int，编码时的最大 token 数
#   batch_size       : int，编码时的批次大小
#   use_fp16         : bool，是否使用 fp16 加速编码
#   faiss_type       : str|None，FAISS 索引类型；"Flat"=精确，"IVF"=近似（速度快）
#   bm25_backend     : str，BM25 后端，"bm25s" 或 "pyserini"
index_builder = Index_Builder(  # Index_Builder: 初始化索引构建器，尚未开始编码
    retrieval_method = config["retrieval_method"],    # str: "bge"
    model_path       = config["retrieval_model_path"],  # str: 编码模型路径
    corpus_path      = config["corpus_path"],         # str: JSONL 语料路径
    save_dir         = INDEX_DIR,                     # str: 索引保存目录（Dense 直接在此目录下生成 bge_Flat.index 文件）
    max_length       = config["retrieval_query_max_length"],  # int: 编码最大 token 数（512，与模型窗口对齐）
    batch_size       = config["retrieval_batch_size"],  # int: 编码批次大小
    # use_fp16 : 推理时是否使用半精度浮点（FP16）
    #   True  → 显存减半、速度约快 1.5~2 倍，轻微精度损失（实际检索效果几乎无差别）
    #   False → FP32 全精度（basic_config.yaml 默认值）
    #   有现代 GPU（RTX 20 系及以上）时建议在 YAML 中设置 retrieval_use_fp16: true
    use_fp16         = config["retrieval_use_fp16"],    # bool: 是否使用 FP16 半精度加速（默认 False）
    # faiss_type : FAISS 索引类型字符串（索引工厂格式），决定向量存储结构和搜索算法
    #   参数直接编码在字符串里，无需单独传参：
    #   "Flat"          → 暴力精确检索，无近似误差，无额外参数，语料 < 10 万条时推荐
    #   "IVF{nlist},Flat" → 聚类分桶近似检索，语料 10 万~百万条时使用
    #     nlist : 聚类桶数，通常取 4*sqrt(N)（N 为语料总条数）
    #     示例：语料 10 万条 → "IVF1280,Flat"（4*sqrt(100000)≈1265，取整）
    #     检索时还需在 DenseRetriever 初始化后、batch_search 前设置 nprobe：
    #       retriever = DenseRetriever(config)
    #       retriever.index.nprobe = 64  # 设置搜索桶数
    #       results = retriever.batch_search(queries)
    #     nprobe 含义：每次检索只搜最近的 nprobe 个桶（默认 1，极快但召回率很低）
    #       nprobe=1          → 极快，召回率低，容易漏结果
    #       nprobe=32~64      → 速度和精度均衡，常用推荐值
    #       nprobe=nlist      → 等价于 Flat 精确搜索，最准但失去速度优势
    #     建议：nprobe 设为 nlist 的 5%~10%（如 nlist=1280 → nprobe=64~128）
    #   "HNSW{M}"       → 图结构近似检索，速度极快，适合大规模生产环境
    #     M : 每个节点的出边数，越大精度越高内存越大，建议 32~64
    #     示例："HNSW32"
    #     构建时可设置 builder.index.hnsw.efConstruction（默认 40，建议 200）
    #   注意：faiss_type 不是 Config 字段，无法在 YAML 中配置，只能在此处直接传参
    #   索引文件名格式：{retrieval_method}_{faiss_type}.index（如 bge_Flat.index）
    faiss_type       = "Flat",   # str: 精确暴力检索（语料小时推荐；大规模可换 IVF/HNSW）
)

print("开始构建 FAISS 索引（首次运行需下载模型，耗时较长）...")  # 打印构建开始提示
# index_builder.build_index() : 执行编码 + 构建 + 保存索引
# 来自源码 build_dense_index()：
#   index_save_path = os.path.join(save_dir, f"{retrieval_method}_{faiss_type}.index")
# 即生成文件名格式为 <method>_<faiss_type>.index（如 bge_Flat.index），扩展名是 .index 不是 .faiss
# 同时生成 emb_{retrieval_method}.memmap（中间嵌入文件，可删除节省空间）
index_builder.build_index()  # 执行编码→构建→保存三步，首次运行需下载模型，耗时较长

print(f"\n索引构建完成！索引文件保存在: {INDEX_DIR}")  # 打印索引保存目录
# 构建完成后，更新 config 的 index_path 指向实际文件
# 来自源码 Index_Builder.build_dense_index()：
#   index_save_path = os.path.join(save_dir, f"{retrieval_method}_{faiss_type}.index")
# 即文件名格式为 <retrieval_method>_<faiss_type>.index（如 bge_Flat.index）
# 注意：扩展名是 .index，不是 .faiss
import glob  # 文件通配符查找模块
# glob.glob(pattern) : 查找匹配 pattern 的所有文件路径，返回 List[str]
index_files = glob.glob(f"{INDEX_DIR}/*.index")  # 匹配 .index 文件，不是 .faiss
if index_files:  # 若找到索引文件则取第一个
    actual_index_path = index_files[0]  # str: 实际生成的索引文件路径（如 bge_Flat.index）
    print(f"发现索引文件: {actual_index_path}")  # 打印索引文件完整路径
else:  # 未找到则使用配置中的占位路径并提示
    actual_index_path = config["index_path"]  # str: fallback，使用 YAML 中的占位路径
    print("未找到 .index 文件，请检查 Index_Builder 是否执行成功")  # 提示用户排查构建失败原因

### 4.3 初始化 DenseRetriever

In [ ]:
# flashrag.retriever.DenseRetriever : 稠密向量检索器
# 初始化时调用 load_index()，要求 config["index_path"] 指向的文件必须已存在
from flashrag.retriever import DenseRetriever  # 稠密向量检索器，基于 FAISS 索引

# 用实际的索引文件路径更新 Config（覆盖 YAML 中的占位路径）
config_with_index = Config(config_path, {"index_path": actual_index_path})  # Config: 覆盖 index_path 的配置对象

# DenseRetriever(config) : 加载编码模型 + 加载 FAISS 索引
# 作用：接受用户 query，编码为向量，在 FAISS 索引中搜索最相似的 Top-K 文档并返回
# 关键 Config 字段（来自源码 update_additional_setting）：
#   以下字段均来自 Config 三层合并结果，未在自定义 YAML 中设置的字段使用
#   basic_config.yaml 默认值，各字段独立覆盖，不会相互冲突
#   retrieval_model_path      : str，编码模型路径
#   corpus_path               : str，JSONL 语料路径（用于 id→文本 映射）
#   index_path                : str，FAISS 索引文件路径
#   retrieval_batch_size      : int，编码批次大小
#   retrieval_topk            : int，检索返回的 Top-K 文档数
#   use_reranker              : bool，是否在检索后自动重排序
#   retrieval_query_max_length: int，查询最大 token 数
#   retrieval_pooling_method  : str，向量池化方式
#     "cls"（默认）→ 取 [CLS] 位置向量，BGE 系列官方推荐，当前模型适用
#     "mean"      → 对所有 token 向量取平均，E5 系列通常使用
#     未在自定义 YAML 中设置，默认 "cls" 正好适合 BGE 模型，无需手动配置
#   use_sentence_transformer  : bool，是否使用 SentenceTransformer 编码器
#     False（默认）→ 用 HuggingFace AutoModel 直接编码，FlashRAG 自己做池化
#                   （池化方式由 retrieval_pooling_method 控制）
#     True        → 改用 SentenceTransformer 库，池化自动处理，
#                   忽略 retrieval_pooling_method
#     对 bge-small-zh-v1.5 两种方式效果基本相同，默认 False 即可
retriever = DenseRetriever(config_with_index)  # DenseRetriever: 加载编码模型与 FAISS 索引

print(f"DenseRetriever 初始化完成！")                         # 打印初始化成功提示
print(f"  索引路径: {config_with_index['index_path']}")        # 打印加载的索引文件路径
print(f"  Top-K   : {config_with_index['retrieval_topk']}")    # 打印每次检索返回的文档数

## 五、检索器

### 5.1 DenseRetriever — 稠密向量检索

> **注意（来自官方源码）**：检索器的批量检索方法名为 `batch_search()`，
> 单条检索方法名为 `search()`。

In [ ]:
# ============================================================
# DenseRetriever 批量检索
# ============================================================

# test_queries : List[str]，批量查询文本列表，长度 N
test_queries = [                              # List[str]: 批量查询文本列表，长度 N=3
    "RAG 技术的核心思想是什么？",              # str: 第 1 条查询
    "FlashRAG 框架支持哪些生成器后端？",       # str: 第 2 条查询
    "FLARE 算法的检索策略",                   # str: 第 3 条查询
]

# retriever.batch_search(query, num, return_score) : 批量向量检索
# 参数（来自源码 batch_search 装饰器链 cache_manager → rerank_manager → _batch_search）：
#   query        : List[str]，查询文本列表，长度 N
#   num          : int|None，返回文档数，None 时使用 config["retrieval_topk"]
#   return_score : bool，是否同时返回相似度分数，默认 False
# 返回值（return_score=False 时）:
#   results : List[List[Dict]]，shape: (N, topk)
#     外层 List 长度 = N（查询数量）
#     内层 List 长度 = topk（Top-K 文档数）
#     按相关性分数降序排列，docs[0] 为最相关文档，docs[-1] 为 Top-K 中最不相关的
#     排序依据取决于检索类型：
#       Dense  → 向量内积/余弦相似度，FAISS 检索时直接返回有序结果
#       BM25   → BM25 分数，由 bm25s 检索后排序
#       Reranker 启用时 → CrossReranker 重打分后重新排序，原始检索顺序会改变
#     注意：若语料总条数 < topk，内层 List 长度小于 topk，不会补 None 或报错
#     每个 Dict 包含 "id"（str）、"contents"（str）、"title"（str，若有）
#     注意：score 字段默认不在 Dict 中，直接访问 doc["score"] 会报 KeyError
#       - use_reranker=False（默认）：doc 只有 id/contents/title，无 score
#       - use_reranker=True：rerank_manager 装饰器在检索后自动重排序，
#         框架可能将重排分数写入 doc["score"]，但此行为不稳定，不建议依赖
#       需要分数时，统一使用 return_score=True，通过独立的 scores 列表获取：
#         results, scores = retriever.batch_search(queries, return_score=True)
#         scores[i][j]  # float: 第 i 条查询第 j 篇文档的相似度分数
# 返回值（return_score=True 时）:
#   返回一个元组 (results, scores)，需用解包接收：
#     results, scores = retriever.batch_search(queries, return_score=True)
#   results : List[List[Dict]]，shape: (N, topk)，文档列表，与 False 时完全相同
#   scores  : List[List[float]]，shape: (N, topk)，每篇文档对应的相似度分数
#     分数含义取决于索引类型：Flat 索引返回向量内积；若向量已归一化则等价于余弦相似度
#     scores[i][j] 对应第 i 条查询的第 j 篇文档的分数，值越大越相关
#   适用场景：分数阈值过滤、多路检索 RRF 融合、调试检索质量、自定义重排序逻辑
results = retriever.batch_search(test_queries)  # List[List[Dict]]: 批量检索结果，shape (N, topk)

# 遍历每条查询及其对应的检索结果，query 为 str，docs 为 List[Dict]
for query, docs in zip(test_queries, results):  # query: str 查询文本，docs: List[Dict] 对应检索结果
    print(f"\n{'='*55}")                             # 打印分隔线，区分不同查询
    print(f"查询: {query}")                             # 打印当前查询文本
    print(f"检索到 {len(docs)} 条文档:")              # 打印返回文档数量
    for rank, doc in enumerate(docs, 1):              # rank 从 1 开始，表示排名
        # doc["contents"] : str，chunk 正文；doc["id"] : str，文档 id
        print(f"  [{rank}] id={doc['id']:10s} | {doc['contents'][:55]}...")  # 打印排名、id 和内容前缀

In [ ]:
# retriever.search(query, num, return_score) : 单条检索（非批量）
# 参数:
#   query        : str，单条查询文本
#   num          : int|None，返回文档数，None 时使用 retrieval_topk
#   return_score : bool，是否返回分数，默认 False
# 返回值（return_score=False）: List[Dict]，长度 = topk
# single_result : List[Dict]，长度 = topk，每个 Dict 含 id/contents/title
single_result = retriever.search("检索增强生成")  # List[Dict]: 单条检索结果，长度 = topk

print("单条检索结果:")  # 打印标题行
# rank 从 1 开始，表示相关性排名；doc 为 Dict（id/contents/title）
for rank, doc in enumerate(single_result, 1):  # rank: int 排名，doc: Dict（id/contents/title）
    print(f"  [{rank}] {doc['contents'][:60]}...")  # 打印排名和文档内容前 60 字

### 5.2 BM25Retriever — 稀疏关键词检索

> **注意 1**：FlashRAG 官方 BM25 后端只支持 `"bm25s"` 和 `"pyserini"`。
> `"rank_bm25"` 不是有效的 backend 值，会导致运行错误。

> **注意 2（来自官方源码）**：`BM25Retriever` 初始化时调用 `bm25s.BM25.load(index_path)`，
> 与 `DenseRetriever` 一样，**不会自动构建索引**，必须先用 `Index_Builder` 构建。
> `Index_Builder` 对 BM25 会把索引保存在 `{INDEX_DIR}/bm25/` 子目录下（不是单个文件）。
> 即传入 `save_dir=INDEX_DIR`，实际生成路径为 `flashrag_workspace/index/bm25/`。
> 来自源码：`self.save_dir = os.path.join(save_dir, "bm25")`，框架自动追加 `/bm25` 后缀。

In [ ]:
# flashrag.retriever.BM25Retriever : 基于 BM25 的稀疏检索器
# 无需 GPU，无需向量编码，适合精确关键词匹配
from flashrag.retriever import BM25Retriever  # BM25 稀疏关键词检索器

# ---- 第一步：先用 Index_Builder 构建 BM25 索引（与 Dense 相同，不能跳过）----
# 来自源码 build_bm25_index_bm25s()：
#   self.save_dir = os.path.join(self.save_dir, "bm25")  # 实际保存在 save_dir/bm25/ 子目录
# 所以 BM25 索引路径 = {INDEX_DIR}/bm25/
bm25_index_dir = f"{INDEX_DIR}/bm25"  # str: BM25 索引实际路径（构建后自动创建）
bm25_builder = Index_Builder(  # Index_Builder: BM25 索引构建器，指定 retrieval_method="bm25" 触发 BM25 分支
    retrieval_method = "bm25",                  # str: 触发 BM25 索引构建分支
    model_path       = "",                       # str: BM25 不需要编码模型，传空字符串
    corpus_path      = config["corpus_path"],    # str: JSONL 语料路径
    save_dir         = INDEX_DIR,               # str: 索引保存根目录（实际生成在 INDEX_DIR/bm25/）
    max_length       = 128,                      # int: BM25 不用此参数，但接口需要传
    batch_size       = 256,                      # int: BM25 不用此参数，但接口需要传
    use_fp16         = False,                    # bool: BM25 不用此参数
    bm25_backend     = "bm25s",                 # str: 指定 bm25s 后端
)
print("构建 BM25 索引...")       # 打印构建开始提示
bm25_builder.build_index()  # 执行构建，结果保存在 INDEX_DIR/bm25/ 目录
print(f"BM25 索引保存在: {bm25_index_dir}")  # 打印 BM25 索引目录路径

# ---- 第二步：初始化 BM25Retriever（加载已构建的索引）----
# 来自源码 BM25Retriever.load_model_corpus()：
#   self.searcher = bm25s.BM25.load(self.index_path, mmap=True, load_corpus=False)
# 即 index_path 必须指向已构建的 BM25 索引目录
config_bm25 = Config(config_path, {  # Config: 覆盖 BM25 检索相关字段的配置对象
    "retrieval_method": "bm25",        # str: 使用 BM25 稀疏检索
    "index_path":       bm25_index_dir, # str: 指向 Index_Builder 实际构建的目录
    # bm25_backend 有效值（来自源码 BM25Retriever.load_model_corpus）：
    #   "bm25s"    : 纯 Python 实现（推荐，默认值）
    #   "pyserini" : 基于 Java Lucene（需要 Java 环境）
    "bm25_backend":     "bm25s",        # str: BM25 后端
})                                      # Config 初始化结束

# BM25Retriever(config) : 加载预构建的 BM25 索引（不自动构建！）
bm25_retriever = BM25Retriever(config_bm25)  # BM25Retriever: 加载预构建的 BM25 索引

# batch_search(queries) : 批量 BM25 检索（方法名与 DenseRetriever 相同）
# 返回值同 DenseRetriever：List[List[Dict]]，shape: (N, topk)
bm25_results = bm25_retriever.batch_search(["BM25 稀疏检索", "FAISS 向量索引"])  # List[List[Dict]]: 批量检索结果

print("BM25 检索结果:")  # 打印标题行
# 遍历查询列表及对应的检索结果，query 为 str，docs 为 List[Dict]
for query, docs in zip(["BM25 稀疏检索", "FAISS 向量索引"], bm25_results):  # query: str，docs: List[Dict]
    print(f"\n  查询: {query}")  # 打印当前查询文本
    for rank, doc in enumerate(docs, 1):  # rank 从 1 开始，表示相关性排名
        print(f"    [{rank}] {doc['contents'][:55]}...")  # 打印排名和文档内容前 55 字

### 5.3 多路混合检索（MultiRetrieverRouter）

FlashRAG 通过 `use_multi_retriever: True` 启用多路检索，
内部使用 `MultiRetrieverRouter` 管理多个子检索器。

> 注意：FlashRAG 当前版本不存在 `HybridRetriever` 独立类，
> 混合检索通过 `multi_retriever_setting` 配置项实现。

In [ ]:
# flashrag.utils.get_retriever : 根据配置自动选择并实例化检索器
# 当 config["use_multi_retriever"] = True 时，返回 MultiRetrieverRouter 实例
from flashrag.utils import get_retriever  # 检索器工厂函数，根据 Config 自动选择检索器类型

# 多路混合检索 Config
# multi_retriever_setting 结构（来自 basic_config.yaml 和 config.py 源码）：
#   merge_method    : str，多路检索结果的融合方式
#     "concat" : 直接拼接去重，不重新排序，顺序保持原样
#               优点：最简单最快；缺点：忽略两路结果的相对质量
#     "rrf"    : Reciprocal Rank Fusion（倒数排名融合）
#               只看排名不看分数：RRF_score = Σ 1/(k + rank_i)，k 通常取 60
#               两路都召回的文档得分叠加天然排前，不受稠密/稀疏分数量纲差异影响
#               鲁棒性强，实际效果通常最好，生产环境首选
#     "rerank" : 多路合并后再过一遍 CrossReranker 精排
#               精度最高，但需加载 reranker 模型，推理耗时最长、资源消耗最大
#     选型建议：快速原型 → concat；生产环境 → rrf；极高精度要求 → rerank
#   topk            : int，融合后保留的文档数（仅 rrf/rerank 模式有效）
#   retriever_list  : List[Dict]，每个子检索器的独立配置
#
# 说明（来自 Config._set_additional_key 源码）：
#   Config 在 _set_additional_key() 中会自动为每个子配置补全下列缺失字段：
#     retrieval_use_fp16, retrieval_query_max_length, faiss_gpu, retrieval_topk,
#     retrieval_batch_size, use_reranker, rerank_model_name, rerank_model_path,
#     retrieval_cache_path, save_retrieval_cache=False, use_retrieval_cache=False
#   因此不必手动填写这些字段；但在自行传 dict 给子检索器时（不经过 Config），
#   仍需手动补全。此处保留 _sub_base 仅作为明确声明，增强可读性。
_sub_base = {                         # 显式声明基础字段（Config 也会自动补全）
    "save_retrieval_cache": False,    # bool: 不保存检索缓存
    "use_retrieval_cache":  False,    # bool: 不使用检索缓存
    "retrieval_cache_path": None,     # str|None
    "use_reranker":         False,    # bool: 子检索器内部不启用 reranker
    "retrieval_model_path": None,     # str|None（Dense 需覆盖为实际路径）
}
config_multi = Config(config_path, {  # Config: 启用多路混合检索的配置对象
    "use_multi_retriever": True,                   # bool: 启用多路检索
    "multi_retriever_setting": {                   # Dict: 多路检索设置字典
        "merge_method": "rrf",                     # str: RRF 融合排名
        "topk": 5,                                 # int: 融合后保留 Top-5
        "retriever_list": [                        # List[Dict]: 子检索器配置列表
            {                                      # Dict: 子检索器 1——稠密检索
                **_sub_base,                       # 展开基础字段（save_retrieval_cache 等）
                "retrieval_method":     "bge",      # str: 稠密检索方式
                "retrieval_topk":       10,         # int: 子检索器各取 Top-10 候选
                "retrieval_model_path": "BAAI/bge-small-zh-v1.5",  # str: 编码模型路径
                # index_path 指向 Index_Builder 生成的 .index 文件
                # 格式：<retrieval_method>_<faiss_type>.index（如 bge_Flat.index）
                "index_path":           actual_index_path,  # str: Dense 索引文件路径
                "corpus_path":          config["corpus_path"],  # str: JSONL 语料路径
            },                                     # end of 子检索器 1 配置
            {                                      # Dict: 子检索器 2——BM25 稀疏检索
                **_sub_base,                       # 展开基础字段
                "retrieval_method":     "bm25",    # str: BM25 稀疏检索方式
                "retrieval_topk":       10,         # int: 子检索器各取 Top-10 候选
                # BM25 索引路径 = Index_Builder 的 {save_dir}/bm25/ 子目录
                # 来自源码 build_bm25_index_bm25s：self.save_dir = os.path.join(save_dir, "bm25")
                "index_path":           bm25_index_dir,   # str: {INDEX_DIR}/bm25/
                "corpus_path":          config["corpus_path"],  # str: JSONL 语料路径
                "bm25_backend":         "bm25s",           # str: BM25 后端
            },                                     # end of 子检索器 2 配置
        ]                                          # end of retriever_list
    },                                             # end of multi_retriever_setting
    "index_path": actual_index_path,  # str: 主检索器（bge）的索引文件路径
})                                                 # Config 初始化结束

# get_retriever(config) :
# 当 use_multi_retriever=True 时，实例化 MultiRetrieverRouter
# MultiRetrieverRouter 内部同时持有所有子检索器，batch_search 时并发检索再融合
multi_retriever = get_retriever(config_multi)  # MultiRetrieverRouter: use_multi_retriever=True 时返回此类实例

# batch_search 方法在 MultiRetrieverRouter 中同样可用
multi_results = multi_retriever.batch_search(["混合检索的优势"])  # List[List[Dict]]: RRF 融合后的结果
# 返回值: List[List[Dict]]，shape: (N, topk)，已按 RRF 融合排名

print("多路混合检索结果（RRF 融合）:")  # 打印标题行
# rank 从 1 开始，doc 为 Dict（id/contents/title）；multi_results[0] 为第一条查询的结果
for rank, doc in enumerate(multi_results[0], 1):  # rank: int 排名，doc: Dict（id/contents/title）
    print(f"  [{rank}] id={doc['id']:10s} | {doc['contents'][:55]}...")  # 打印排名和文档内容

## 六、重排序器（CrossReranker）

FlashRAG 重排序器实际类名（来自 `flashrag/retriever/reranker.py` 源码）：
- `CrossReranker`：CrossEncoder 精排，将 `(query + doc)` 拼接后送入模型，query 与 doc 充分交互，精度高；
  速度较慢，文档向量无法预先缓存，适合追求精度的场景，**绝大多数情况首选此类**
- `BiReranker`：BiEncoder 精排，query 和 doc 仍然分别编码再计算相似度，原理与 `DenseRetriever` 相同；
  可以预编码文档向量、速度较快，但精度低于 `CrossReranker`（无 query-doc 交互）；
  与 `DenseRetriever` 的区别在于可以换用更大的 BiEncoder 模型，只处理 Top-K 候选，实际中较少使用

`get_reranker(config)` 根据模型架构自动选择正确的类（推荐使用）：
读取 `rerank_model_name` / `rerank_model_path`，判断模型是 CrossEncoder 还是 BiEncoder，自动返回对应实例。
好处是换模型时只需修改配置，无需改代码；直接写 `CrossReranker(config)` 也可以，效果完全相同（需明确知道模型架构）。

**用法对比：**
```python
from flashrag.retriever import get_reranker, CrossReranker, BiReranker

# 推荐：工厂函数，自动根据 rerank_model_name 判断架构
reranker = get_reranker(config)

# 等价写法（明确知道模型是 CrossEncoder 时）
reranker = CrossReranker(config)

# 等价写法（明确知道模型是 BiEncoder 时）
reranker = BiReranker(config)
```

In [ ]:
# flashrag.retriever.CrossReranker : 基于 CrossEncoder 的重排序器
# 实际类名来自 flashrag/retriever/reranker.py 源码（非 CrossEncoderReranker）
#
# ---- 为什么有检索还需要重排序 ----
# 检索（DenseRetriever / 混合检索）和重排序解决的是不同阶段的不同问题：
#
# 检索阶段（粗排）— 解决"找得到"的问题
#   稠密路：bge-small-zh-v1.5（BiEncoder）把 query 和 doc 分别独立编码成向量，
#           提前建好 FAISS 索引，检索时只做向量距离计算，毫秒级从百万文档中筛出候选
#   BM25 路：词频统计匹配，无需编码模型，速度极快
#   混合路：稠密 + BM25 并行，RRF 融合 → Top-K 候选
#   缺点：query 和 doc 编码时互不知晓对方内容，语义交互浅，容易召回表面相关但无用的文档
#
# 重排序阶段（精排）— 解决"排得准"的问题
#   CrossReranker 使用 CrossEncoder（bge-reranker-base）把 (query + doc) 拼在一起送入模型，
#   底层模型是 BERT 类编码器（如 bge-reranker-base 基于 BERT-base），
#   拼接格式：[CLS] query [SEP] doc [SEP]，作为一个完整序列送入模型，
#   所有 token 在每一层 Self-Attention 中相互可见，
#   query 中的每个 token 都能 attend 到 doc 中的每个 token（反之亦然），充分交互；
#   打分方式：取 [CLS] 位置的最终隐藏向量（BERT 用它代表整句语义），
#   经过 Linear(hidden_size→1) 映射为单个相关性标量，
#   再通过 Sigmoid 压缩到 [0, 1]，数值越大表示越相关（与 BiEncoder 点积打分本质不同）
#   缺点：每对都要过一次模型，只能处理少量候选（Top-K），无法直接对全库使用
#
# 两阶段组合：
#   全库（百万文档）→ 检索粗筛（快）→ Top-50 候选
#                                    → 重排序精排（准）→ Top-3 → 生成器
#   用检索的速度 + 用重排序的精度，是工业界 RAG 的标准范式
from flashrag.retriever import CrossReranker  # CrossEncoder 精排重排序器（官方类名）

# CrossReranker(config) : 初始化重排序器
# 从 Config 中读取（来自源码 BaseReranker.__init__）：
#   config["rerank_model_name"] : str，重排序模型别名
#   config["rerank_model_path"] : str，重排序模型路径
#   config["rerank_topk"]       : int，重排序后保留的文档数
#   config["rerank_max_length"] : int，输入最大 token 数
#   config["rerank_batch_size"] : int，推理批次大小
#   config["device"]            : str，设备（"cuda"/"cpu"，Config 自动设置）
config_rerank = Config(config_path, {"use_reranker": True})  # Config: 启用重排序的配置对象
reranker = CrossReranker(config_rerank)  # CrossReranker: 加载 CrossEncoder 重排序模型

print(f"CrossReranker 初始化完成！")                         # 打印初始化成功提示
print(f"  模型路径: {config_rerank['rerank_model_path']}")    # 打印 reranker 模型路径
print(f"  保留 Top-K: {config_rerank['rerank_topk']}")        # 打印重排序后保留的文档数

In [ ]:
# ============================================================
# 检索 + 重排序完整流程
# ============================================================
rerank_queries = ["RAG 与传统问答的区别", "混合检索的优势"]  # List[str]: 查询列表

# Step 1: 粗排检索，获取候选文档
# all_candidates : List[List[Dict]]，shape: (N, retrieval_topk)
all_candidates = retriever.batch_search(rerank_queries)  # List[List[Dict]]: 粗排候选文档，shape (N, retrieval_topk)

print("粗排候选文档数:", [len(d) for d in all_candidates])  # 打印每条查询的候选文档数量

# Step 2: 精排重排序
# reranker.rerank(query_list, doc_list, batch_size, topk) :
# 参数（来自源码 BaseReranker.rerank）：
#   query_list : str | List[str]，查询文本（支持单条 str 或列表）
#   doc_list   : List[str_or_dict] | List[List[str_or_dict]]，候选文档
#                若 query_list 为 str，doc_list 可以是 List[Dict]（自动包装）
#                若 query_list 为 List，doc_list 必须是 List[List[Dict]]
#   batch_size : int|None，推理批次大小，None 使用 config 中的值
#   topk       : int|None，保留文档数，None 使用 config 中的值
# 返回值（重要！来自源码）：
#   (final_docs, final_scores) : Tuple[List[List[Dict]], List[List[float]]]
#     final_docs   : List[List[Dict]]，重排序后的文档列表，已按相关性降序排列
#                    shape: (N, topk)
#     final_scores : List[List[float]]，对应的 CrossEncoder 分数
#                    shape: (N, topk)，值越高越相关（logit 值，无固定范围）
reranked_docs, rerank_scores = reranker.rerank(rerank_queries, all_candidates)  # Tuple[List[List[Dict]], List[List[float]]]

print("\n--- 重排序结果 ---")  # 打印结果区标题
# 同时遍历查询、精排文档、精排分数，zip 长度 = N（查询数量）
for q, docs, scores in zip(rerank_queries, reranked_docs, rerank_scores):  # q: str，docs: List[Dict]，scores: List[float]
    print(f"\n查询: {q}")                       # 打印当前查询文本
    print(f"精排后保留 {len(docs)} 条文档:")     # 打印精排后的文档数量
    for rank, (doc, score) in enumerate(zip(docs, scores), 1):  # rank 从 1 开始
        # score : float，CrossEncoder logit 分数，越大越相关
        print(f"  [{rank}] score={score:.4f} | {doc['contents'][:50]}...")  # 打印排名、分数和内容

## 七、生成器

### 7.1 OpenaiGenerator — OpenAI 兼容 API 生成器

In [ ]:
# flashrag.generator.OpenaiGenerator : OpenAI 兼容 API 生成器
# 内部使用 AsyncOpenAI SDK（来自源码 openai_generator.py）
# 支持所有兼容 OpenAI 接口规范的服务，实际使用的是 /v1/chat/completions（现代接口）：
#   /v1/chat/completions（主流，实际使用）：
#     输入为消息列表 [{'role': 'user', 'content': '...'}]，支持多轮对话、
#     system prompt、assistant 历史；GPT-3.5/4、Qwen、LLaMA-3 等现代模型均使用此接口
#   /v1/completions（旧版，已废弃）：
#     输入为纯文本 prompt，模型直接续写；GPT-3 时代接口，
#     OpenAI 官方已标注 deprecated，仅 text-davinci-003 等老模型仍支持，现代模型不推荐使用
from flashrag.generator import OpenaiGenerator  # OpenAI 兼容 API 生成器（异步批量请求）

# OpenaiGenerator(config) : 使用 Config 初始化
# 注意：config_with_index 在「五、检索器」小节（5.1 DenseRetriever）中定义，
#       需先运行该小节的 cell 后此处才可正常使用；
# 关键字段（来自源码 update_base_setting）：
#   config["generator_model"]     : str，API 模型 ID
#   config["openai_setting"]      : Dict，直接传给 AsyncOpenAI(**openai_setting)
#   config["generation_params"]   : Dict，生成超参数（max_tokens、temperature 等）
#   config["generator_batch_size"]: int，并发请求数
generator = OpenaiGenerator(config_with_index)  # OpenaiGenerator: 使用配置初始化 API 生成器

question = "RAG 技术的三个发展阶段是什么？"  # str: 用户问题

# 先检索相关文档
top_docs = retriever.batch_search([question])[0]  # List[Dict]: Top-K 候选文档

# 手动构造 Prompt（chat 模式，使用消息列表）
# context : str，将 Top-K 文档拼接为带编号的多段文本，作为参考上下文
context = "\n\n".join(          # str: 将 Top-K 文档拼接为带编号的多段文本，作为参考上下文
    f"[文档 {i+1}] {doc['contents']}" for i, doc in enumerate(top_docs)  # i: int 序号，doc: Dict 文档
)
# messages : List[Dict]，OpenAI chat 格式的消息列表
messages = [                    # List[Dict]: OpenAI chat 格式消息列表
    {"role": "system",  "content": "你是一个专业的知识问答助手，请根据参考文档准确回答。"},  # Dict: 系统角色消息
    {"role": "user",    "content": f"参考文档：\n{context}\n\n问题：{question}"},            # Dict: 用户消息（含文档上下文和问题）
]

# generator.generate(input_list, batch_size, return_scores, **params) :
# 参数（来自源码 _generate_async）：
#   input_list    : List[str | dict | List[dict]]，输入列表
#     List[str]         → completion 模式（直接生成）
#     List[List[dict]]  → chat 模式（messages 列表，含 role/content）
#   return_scores : bool，是否返回 logprobs（对数概率），默认 False
#     logprobs：模型对每个生成 token 的对数概率 ln(P(token))，范围 (-∞, 0]，
#     越接近 0 表示模型对该 token 越确定；常见用途：
#       - 置信度估计：logprob 越低（越负）说明生成越不确定，可过滤低质量答案
#       - 困惑度计算：exp(-mean(logprobs))，评估生成文本流畅度
#       - 多候选打分：比较不同答案的总 logprob 来选最优
# 返回值:
#   answers : List[str]，生成的回答列表，长度 = len(input_list)
#   （若 return_scores=True，返回 (List[str], List[scores])）
# 传入 List[List[Dict]]（chat 模式）
answers = generator.generate([messages])  # List[str]: 生成的回答列表，长度 1

print(f"问题: {question}")          # 打印问题文本
print(f"\n回答: {answers[0]}")      # 打印生成的回答（answers[0] 为第一条，也是唯一一条）

### 7.2 HFCausalLMGenerator — 本地 HuggingFace 模型生成器

当 `framework="hf"` 时，`get_generator()` 自动实例化 `HFCausalLMGenerator`
（针对 CausalLM 架构的模型，如 Qwen、LLaMA 等）。

> 注意：官方类名为 `HFCausalLMGenerator`。

In [ ]:
# flashrag.utils.get_generator(config) :
# 根据 config["framework"] 自动选择正确的生成器类（推荐使用此函数而非直接导入类）
#   "openai" → OpenaiGenerator
#   "hf"     → HFCausalLMGenerator（CausalLM）或 EncoderDecoderGenerator（T5/BART）
#   "vllm"   → VLLMGenerator
#   "fschat" → FastChatGenerator
from flashrag.utils import get_generator  # 生成器工厂函数，根据 framework 自动选择生成器类型

# 切换为本地模型只需修改 framework 和 generator_model_path
config_hf = Config(config_path, {   # Config: 指定本地 HF 模型的配置对象
    "framework":            "hf",                   # str: 本地 HuggingFace 模型
    # generator_model_path : 本地模型目录路径（需提前下载）
    # 若 generator_model_path 为 None，Config 会在 model2path 中查找 generator_model 对应路径
    "generator_model_path": "/path/to/local/Qwen2.5-1.5B-Instruct",  # str: 替换为实际路径
    "generation_params": {           # Dict: HF 模型生成超参数
        # HF 模型使用 max_new_tokens（而非 max_tokens），注意区分
        "max_new_tokens": 256,           # int: 最大新生成 token 数
        "temperature":    0.1,            # float: 采样温度
        "do_sample":      True,           # bool: 解码策略
        # do_sample=False（贪婪解码）：每步选概率最高的 token，确定性输出，
        #   相同输入永远得到相同结果，速度快，但易陷入重复循环、缺乏多样性
        # do_sample=True（采样）：每步按概率分布随机抽取 token，引入随机性，
        #   结合 temperature/top_p/top_k 控制多样性和质量，适合开放式生成
        #   - temperature<1：压低低概率 token，输出更保守集中（0.1=近似贪婪）
        #   - temperature>1：拉平概率分布，输出更随机多样
        #   - top_p（核采样）：只从累积概率≥p 的最小 token 集合中采样，过滤长尾低质量 token
        #   - top_k：只从概率最高的 k 个 token 中采样
        # RAG 问答场景推荐：do_sample=True + temperature=0.1~0.3，接近确定性但保留少量灵活性
    },                               
})                                   

# 以下代码在本地模型不存在时会报错，仅演示 API 用法
# 取消注释并替换 generator_model_path 后可实际运行

# hf_gen = get_generator(config_hf)     # 自动选择 HFCausalLMGenerator
# answers = hf_gen.generate([messages]) # 与 OpenaiGenerator 完全相同的调用接口

print("HFCausalLMGenerator 使用说明:")   # 打印使用说明标题
print("  1. 修改 generator_model_path 为本地模型目录")  # 步骤 1
print("  2. 取消上方两行注释")                         # 步骤 2
print("  3. Pipeline 及评估代码无需任何修改")            # 步骤 3
# 打印当前 HF 框架和模型路径，供用户确认配置是否正确
print(f"\n  当前 HF 配置: framework={config_hf['framework']}, model_path={config_hf['generator_model_path']}")  # 打印 HF 框架和模型路径，供用户确认

### 7.3 PromptTemplate — 自定义 Prompt 模板

> 注意（来自官方源码）：构造 Prompt 的方法名为 `get_string()`。

In [ ]:
# flashrag.prompt.PromptTemplate : Prompt 模板管理类（来自 flashrag/prompt/base_prompt.py）
# Pipeline 内部使用此类构造 Prompt，也可单独使用
from flashrag.prompt import PromptTemplate  # Prompt 模板管理类，负责填充占位符并构造消息列表

# PromptTemplate(config, system_prompt, user_prompt, ...) :
# 参数（来自源码）：
#   config        : Config 对象
#   system_prompt : str|None，系统角色提示（放入 messages 的 system 消息）
#   user_prompt   : str|None，用户消息模板（含 {question}、{reference} 等占位符）
template = PromptTemplate(              # PromptTemplate: 使用自定义 system/user_prompt 初始化模板
    config_with_index,                 # Config: FlashRAG 配置对象（定义于「五、5.1 DenseRetriever」小节）
    system_prompt="你是一个专业的知识问答助手，请根据文档准确回答，不要编造不在文档中的信息。",  # str: 系统角色提示
    user_prompt="参考文档：\n{reference}\n\n问题：{question}",   # str: 用户消息模板，{reference}/{question} 为占位符
    # reference 占位符：Pipeline 内部将检索文档格式化为字符串后填入
)

# template.get_string(question, retrieval_result, ...) :
# 实际方法名（来自源码 SequentialPipeline.run 中的调用）
# 参数：
#   question         : str，用户问题，填充 {question} 占位符
#   retrieval_result : List[Dict]，检索结果，框架自动拼接为文档文本并填充 {reference}
# 返回值:
#   prompt : str | List[Dict]，构造好的 Prompt
#     若 system_prompt 不为 None，返回 messages 列表（chat 格式）
#     否则返回纯字符串（completion 格式）
demo_docs = retriever.batch_search(["RAG 技术"])[0][:2]  # List[Dict]: 取 Top-2 演示文档
prompt_result = template.get_string(        # 构造 Prompt，返回 str 或 List[Dict]
    question         = "什么是 RAG 技术？",    # str: 用户问题，填充 {question} 占位符
    retrieval_result = demo_docs,              # List[Dict]: 检索结果，填充 {reference} 占位符
)

print("PromptTemplate.get_string() 构造的 Prompt 类型:", type(prompt_result))  # 打印返回值类型
if isinstance(prompt_result, list):  # chat 模式：返回 List[Dict]
    # 遍历消息列表，每条消息含 role（str）和 content（str）
    for msg in prompt_result:  # msg: Dict，含 "role"（str）和 "content"（str）两个字段
        # msg : Dict，含 "role" 和 "content" 字段
        print(f"  [{msg['role']}] {str(msg['content'])[:80]}...")  # 打印角色和内容前 80 字
else:  # completion 模式：返回 str
    print(prompt_result[:200] + "...")  # 打印 Prompt 字符串前 200 字

## 八、Pipeline 实战

FlashRAG Pipeline 一览（来自 `flashrag/pipeline/__init__.py` 源码）：

| 类名 | 来源模块 | 算法 |
|------|---------|------|
| `SequentialPipeline` | `pipeline.py` | 标准 Naive RAG |
| `ConditionalPipeline` | `pipeline.py` | 判断是否需要检索 |
| `FLAREPipeline` | `active_pipeline.py` | 前向迭代检索 |
| `SelfRAGPipeline` | `active_pipeline.py` | 自适应检索 |
| `IterativePipeline` | `active_pipeline.py` | 多轮迭代检索 |
| `IRCOTPipeline` | `active_pipeline.py` | CoT + 迭代检索 |
| `REPLUGPipeline` | `branching_pipeline.py` | 集成多路检索 |

> 注意：当前版本不存在 `HyDEPipeline`（未在 `__init__.py` 的任何导入中列出）。
> HyDE 可通过自定义 Pipeline 手动实现（见 8.4 节）。

---

**SequentialPipeline — 标准 Naive RAG**

**流程**：检索 → 拼接上下文 → 生成，一次性完成，无迭代。

- **优点**：实现最简单，延迟最低，适合绝大多数知识问答场景；推理开销 = 1 次检索 + 1 次生成
- **缺点**：无法处理需要多跳推理的复杂问题；检索质量直接决定最终答案质量
- **适用场景**：单跳事实性问答、文档摘要、FAQ 检索

---

**ConditionalPipeline — 判断是否需要检索**

**流程**：Judger 判断 → 需要检索则走 RAG，否则直接生成。

- **优点**：对不需要外部知识的问题（如简单计算、常识问题）跳过检索，节省延迟和 token 开销
- **缺点**：Judger 判断准确性影响整体效果；需要额外训练或提示工程维护 Judger
- **适用场景**：混合型问答系统（部分问题需检索，部分不需要）

---

**FLAREPipeline — 前向迭代检索（FLARE）**

**流程**：生成时若出现低置信度 token（logprob 低于阈值），暂停生成 → 触发检索 → 补充信息 → 继续生成，可触发多次。

- **优点**：能在生成过程中动态补充知识，适合需要分步引用多段文档的长文本生成
- **缺点**：多次检索导致延迟显著增加；低置信度阈值难以调优
- **适用场景**：长篇文章生成、报告撰写等需要逐段引用不同来源的场景

---

**SelfRAGPipeline — 自适应检索（Self-RAG）**

**流程**：模型输出特殊 reflection token 自主决定是否检索、对哪些检索结果打分并过滤，再生成答案。

- **优点**：检索决策由模型自主完成，灵活性最高；可主动过滤低质量检索结果
- **缺点**：**必须使用专门 fine-tune 过的 Self-RAG 模型**（如 selfrag-llama2-7b），通用模型无法使用；训练成本高
- **适用场景**：追求端到端自适应 RAG、已有 Self-RAG 专用模型的场景

---

**IterativePipeline — 多轮迭代检索**

**流程**：固定迭代 N 轮，每轮：检索 → 生成中间答案 → 用中间答案作为新 query 再次检索，逐步完善最终答案。

- **优点**：每轮检索的 query 越来越精准，适合需要分步拆解的复杂多跳问答
- **缺点**：推理开销是单路的 N 倍；N 需要手动指定，过多轮次收益递减
- **适用场景**：多跳推理（如"A 的老师的母校在哪里"）、需要逐步收集信息的复杂问题

---

**IRCOTPipeline — CoT + 迭代检索（IR-CoT）**

**流程**：思维链（CoT）推理与迭代检索交替进行，每生成一步推理后检索补充依据，直到得出最终答案。

- **优点**：将 CoT 推理能力与外部知识检索深度结合，对复杂推理问题效果显著优于 IterativePipeline
- **缺点**：推理步骤数不固定，延迟难以预测；对生成模型的推理能力要求较高
- **适用场景**：需要多步逻辑推理 + 外部知识的复杂问答（如 HotpotQA、MuSiQue）

---

**REPLUGPipeline — 集成多路检索（REPLUG）**

**流程**：用多个检索结果（Top-K 文档）分别独立生成 K 条候选答案，再按检索相关性分数加权融合（ensemble），选出最终答案。

- **优点**：通过集成多路提高答案稳定性和鲁棒性，减少单路检索噪声的影响
- **缺点**：推理开销是单路的 K 倍（K = retrieval_topk），延迟和 token 消耗大幅增加；不适合实时场景
- **适用场景**：离线批量评估、对准确率要求极高且不在意延迟的场景

### 8.1 准备 Pipeline 输入数据集

In [ ]:
import json  # JSON 序列化模块

# ============================================================
# FlashRAG 数据集 JSONL 格式（每行必填字段）：
#   question       : str，问题文本
#   golden_answers : List[str]，参考答案列表（评估用）
#   id             : str，问题唯一标识（可选）
# ============================================================
# 注意（来自 Config 源码）：
#   dataset_path = data_dir / dataset_name
#   所以数据集文件须位于 {WORKSPACE}/{DATASET_NAME}/test.jsonl
# ============================================================
qa_data = [  # List[Dict]: 测试问答数据集，每项含 id/question/golden_answers 三个字段
    {"id": "q0", "question": "RAG 技术的三个发展阶段是什么？",      # str: 问题 0
     "golden_answers": ["朴素 RAG、高级 RAG、模块化 RAG"]},          # List[str]: 参考答案列表
    {"id": "q1", "question": "FlashRAG 支持哪些生成器后端？",         # str: 问题 1
     "golden_answers": ["OpenaiGenerator、HFCausalLMGenerator、VLLMGenerator"]},  # List[str]: 参考答案列表
    {"id": "q2", "question": "FLARE 算法的核心特点是什么？",           # str: 问题 2
     "golden_answers": ["在生成过程中遇到低置信度 token 时主动触发新一轮检索"]},   # List[str]: 参考答案列表
]

# 保存到正确路径：{WORKSPACE}/{DATASET_NAME}/test.jsonl
# （对应 Config 中 dataset_path = data_dir / dataset_name 的计算结果）
test_data_path = f"{DATA_SUBDIR}/test.jsonl"  # str: 实际数据集文件路径
# 逐行写入 JSONL 文件，每条问答对占一行
with open(test_data_path, "w", encoding="utf-8") as f:  # 文件句柄 f，写入完毕后自动关闭
    for item in qa_data:  # 遍历问答对列表，item 为 Dict（id/question/golden_answers）
        f.write(json.dumps(item, ensure_ascii=False) + "\n")  # 序列化为 JSON 行

print(f"数据集写入: {test_data_path}，共 {len(qa_data)} 条")  # 打印写入路径和数量

# flashrag.utils.get_dataset(config) :
# 读取 config["dataset_path"]（= data_dir/dataset_name）目录下的 JSONL 文件
# 返回 Dict[str, Dataset]，key 为 split 名（"test"），value 为 Dataset 对象
from flashrag.utils import get_dataset  # 数据集加载工具函数，读取 JSONL 返回 Dict[str, Dataset]

all_splits   = get_dataset(config_with_index)  # Dict[str, Dataset]: key 为 split 名，value 为 Dataset
test_dataset = all_splits["test"]              # Dataset: 测试集（对应 test.jsonl）
# Dataset 是 FlashRAG 的内置数据集类，封装了 List[Dict] 样本；
#   len(test_dataset)             : int，样本数量（= test.jsonl 行数）
#   test_dataset[i]               : Dict，第 i 条样本，含字段：
#     'id'            : str，样本唯一标识
#     'question'      : str，问题文本
#     'golden_answers': List[str]，标准答案列表（可含多个等价答案）
#   test_dataset.data             : List[Dict]，所有样本的原始列表

print(f"Dataset 加载完成，共 {len(test_dataset)} 条")               # 打印数据集总条数
print(f"  第1条 question: {test_dataset[0]['question']}")         # 打印第一条问题文本

### 8.2 SequentialPipeline — Naive RAG

`SequentialPipeline.run()` 内部调用 `retriever.batch_search()`（来自源码），
再调用 `prompt_template.get_string()` 构造 Prompt，最后调用 `generator.generate()`。

In [ ]:
# flashrag.pipeline.SequentialPipeline : 标准 Naive RAG Pipeline
# 流程（来自源码 SequentialPipeline.run）：
#   retriever.batch_search(questions) → prompt_template.get_string() → generator.generate()
from flashrag.pipeline import SequentialPipeline  # 标准 Naive RAG Pipeline（检索→Prompt→生成）

# SequentialPipeline(config, prompt_template, retriever, generator) :
# 参数（来自源码，均有默认值）：
#   config          : Config，必填
#   prompt_template : PromptTemplate|None，None 时使用 PromptTemplate(config)
#   retriever       : Retriever|None，None 时用 get_retriever(config) 自动创建
#   generator       : Generator|None，None 时用 get_generator(config) 自动创建
naive_pipeline = SequentialPipeline(config_with_index)  # SequentialPipeline: 使用配置自动创建检索器和生成器

# pipeline.run(dataset, do_eval, pred_process_fun) :
#   dataset         : Dataset，输入数据集
#   do_eval         : bool，是否计算评估指标，默认 True
#   pred_process_fun: callable|None，后处理函数
# 返回值:
#   output_dataset  : Dataset，增加了以下字段：
#     "pred"             : str，模型预测答案
#     "retrieval_result" : List[Dict]，检索结果
#     "prompt"           : str|List[Dict]，实际使用的 Prompt
print("运行 Naive RAG Pipeline...")  # 打印运行开始提示
naive_output = naive_pipeline.run(test_dataset, do_eval=False)  # Dataset: 增加了 pred/retrieval_result/prompt 字段

print("\n--- Naive RAG 结果 ---")  # 打印结果区标题
# 遍历输出数据集，每条 item 为 Dict，含 question/pred/retrieval_result 等字段
for item in naive_output:  # item: Dict（question/pred/retrieval_result/golden_answers）
    print(f"\n问题: {item['question']}")         # 打印原始问题文本
    print(f"预测: {item['pred'][:80]}...")        # 打印模型生成回答（前 80 字）

### 8.3 FLAREPipeline — 前向迭代检索

In [ ]:
# flashrag.pipeline.FLAREPipeline : 前向迭代检索 Pipeline
# 来自 flashrag/pipeline/active_pipeline.py
# 流程：生成 → 监控 token 置信度（logprobs）→ 低置信时主动检索 → 融入上下文继续生成
from flashrag.pipeline import FLAREPipeline  # 前向迭代检索 Pipeline（来自 active_pipeline.py）

# FLARE 需要 logprobs，需在 generation_params 中启用
config_flare = Config(config_path, {           # Config: FLARE 专用配置，启用 logprobs
    "index_path": actual_index_path,           # str: FAISS 索引文件路径
    "generation_params": {                     # Dict: 生成超参数，需额外开启 logprobs
        "max_tokens":  512,                    # int: 最大生成 token 数
        "temperature": 0.1,                    # float: 采样温度
        "logprobs":    True,    # bool: 开启 token 概率输出（FLARE 必须启用）
        "top_logprobs": 1,      # int: 每个位置返回 Top-1 logprob
    },                                         # end of generation_params
})                                             # Config 初始化结束

# FLAREPipeline(config, ...) : 初始化，参数同 SequentialPipeline
flare_pipeline = FLAREPipeline(config_flare)   # FLAREPipeline: 前向迭代检索 Pipeline 实例

print("运行 FLARE Pipeline（迭代检索）...")  # 打印运行开始提示
flare_output = flare_pipeline.run(test_dataset, do_eval=False)  # Dataset: 增加了 pred 等字段

print("\n--- FLARE 结果 ---")  # 打印结果区标题
# 遍历输出数据集，item 含 question/pred/retrieval_result 等字段
for item in flare_output:  # item: Dict（question/pred/retrieval_result/golden_answers）
    print(f"\n问题: {item['question']}")    # 打印原始问题文本
    print(f"预测: {item['pred'][:80]}...")  # 打印模型生成回答（前 80 字）

### 8.4 手动实现 HyDE 逻辑

HyDE（Hypothetical Document Embeddings）在 FlashRAG 当前版本中没有独立 Pipeline 类，
可通过两步调用手动实现：
1. 用生成器生成假设文档（Hypothetical Document）
2. 用假设文档的向量代替问题向量进行检索

In [ ]:
# ============================================================
# HyDE 手动实现（两步）
# 来自 HyDE 论文（Gao et al. 2022）思路：
#   Step 1: Query → Generator → Hypothetical Document
#   Step 2: Hypothetical Document → Retriever → Top-K 真实文档
#   Step 3: 真实文档 → Generator → 最终回答
# ============================================================

hyde_question = "什么是 RAG 技术？"  # str: 用户问题

# ---- Step 1: 生成假设文档 ----
# 用 generator 根据问题生成一段假设性的回答文档
hyde_prompt_messages = [                       # List[Dict]: HyDE 专用的假设文档生成 Prompt
    {"role": "user", "content":                # Dict: 用户消息，要求生成假设文档
     f"请写一段关于以下问题的回答段落（不要直接回答问题，而是写成百科知识文档风格）：\n{hyde_question}"  # str: 指示内容
    }
]

# generator.generate([messages]) : 传入 List[List[Dict]]（chat 模式）
# 返回值 : List[str]，长度 1，第一条为假设文档文本
hypo_doc_texts = generator.generate([hyde_prompt_messages])  # List[str]: 长度 1，假设文档文本列表
hypo_doc = hypo_doc_texts[0]  # str: 生成的假设文档文本

print(f"假设文档（前 100 字）: {hypo_doc[:100]}...")  # 打印生成的假设文档内容

# ---- Step 2: 用假设文档检索真实文档 ----
# 直接将假设文档作为查询传入检索器
# retriever.batch_search([hypo_doc]) : 以假设文档作为查询文本进行向量检索
# 返回值 : List[List[Dict]]，取第 0 个（对应唯一一条查询）
hyde_retrieved = retriever.batch_search([hypo_doc])[0]  # List[Dict]: Top-K 真实文档

# ---- Step 3: 基于真实文档生成最终回答 ----
# context : str，将 Top-K 真实文档拼接为带编号的多段文本
context = "\n\n".join(                         # str: 将 Top-K 真实文档拼接为带编号的上下文文本
    f"[文档 {i+1}] {doc['contents']}" for i, doc in enumerate(hyde_retrieved)  # i: int 序号，doc: Dict
)
# final_messages : List[Dict]，包含 system 和 user 两条消息的 chat 格式列表
final_messages = [                             # List[Dict]: 最终回答的 chat 格式消息列表
    {"role": "system", "content": "请根据参考文档准确回答问题。"},                          # Dict: 系统角色消息
    {"role": "user",   "content": f"参考文档：\n{context}\n\n问题：{hyde_question}"},        # Dict: 用户消息（含文档和问题）
]
# final_answers : List[str]，长度 1，对应 final_messages 的生成回答
final_answers = generator.generate([final_messages])  # List[str]: 长度 1，基于真实文档的最终回答

print(f"\n问题: {hyde_question}")                      # 打印原始问题
print(f"检索文档数: {len(hyde_retrieved)}")             # 打印检索到的真实文档数量
print(f"最终回答: {final_answers[0][:120]}...")         # 打印最终回答（前 120 字）

### 8.5 自定义 Pipeline

继承 `BasicPipeline`，重写 `run()` 方法，实现任意检索-生成逻辑。

In [ ]:
# flashrag.pipeline.BasicPipeline : 所有 Pipeline 的基类（来自 pipeline.py 源码）
# 提供 self.retriever、self.generator、self.evaluator、self.prompt_template
from flashrag.pipeline import BasicPipeline  # 所有 Pipeline 的基类，提供 retriever/generator/evaluator 等属性


class RerankRAGPipeline(BasicPipeline):  # 继承 BasicPipeline，自定义「检索→重排序→生成」流程
    """
    带 CrossReranker 精排的自定义 RAG Pipeline。

    流程：
      1. batch_search() 粗排，召回 Top-K 候选文档
      2. CrossReranker.rerank() 精排，筛选 Top-N 文档
      3. get_string() 构造 Prompt，generate() 生成最终回答

    Parameters
    ----------
    config   : Config，FlashRAG 配置对象
    reranker : CrossReranker，已初始化的重排序器
    """

    def __init__(self, config, reranker):  # config: Config 配置对象，reranker: CrossReranker 重排序器
        """
        初始化自定义 Pipeline，注入外部重排序器。

        Parameters
        ----------
        config   : Config，配置对象
        reranker : CrossReranker，重排序器实例
        """
        # super().__init__ 是 BasicPipeline 基类的构造函数，签名为：
        # BasicPipeline(config, prompt_template=None)：两个参数（来自源码验证）
        # 基类 __init__ 只初始化：config、device、evaluator、prompt_template（默认自动创建），
        # 并将 self.retriever = None；不会自动创建 retriever/generator，
        # retriever/generator 的自动初始化由 SequentialPipeline 等子类负责，
        # 因此此处自定义 Pipeline 直接继承 BasicPipeline，需在 run() 中手动使用 self.retriever
        super().__init__(config)        # 调用父类 BasicPipeline.__init__，自动初始化 retriever/generator 等
        self.reranker = reranker  # CrossReranker: 保存重排序器

    def run(self, dataset, do_eval: bool = True, pred_process_fun=None):  # dataset: Dataset 输入数据集
        """
        执行完整「检索 → 重排序 → 生成」流程。

        Parameters
        ----------
        dataset         : Dataset，FlashRAG 格式的问答数据集
        do_eval         : bool，是否计算评估指标，默认 True
        pred_process_fun: callable|None，预测结果后处理函数

        Returns
        -------
        dataset : Dataset，增加了 pred 和 retrieval_result 字段
        """
        # dataset.question : List[str]，数据集所有问题的属性访问（来自 Dataset 源码）
        questions = dataset.question  # List[str]: 所有问题，长度 N

        # ---- Step 1: 批量粗排检索 ----
        # self.retriever.batch_search(questions) : 返回 List[List[Dict]]，shape (N, topk)
        all_candidates = self.retriever.batch_search(questions)  # List[List[Dict]]: 粗排候选文档

        # ---- Step 2: 批量精排重排序 ----
        # self.reranker.rerank(query_list, doc_list) :
        #   返回 (final_docs, final_scores)，final_docs shape: (N, rerank_topk)
        all_reranked, all_scores = self.reranker.rerank(questions, all_candidates)  # (List[List[Dict]], List[List[float]])

        # ---- Step 3: 构造 Prompt 并生成 ----
        # self.prompt_template.get_string(question, retrieval_result) : 构造 Prompt
        # 内部调用 format_reference(retrieval_result) 将 List[Dict] 格式化为可读文本，
        # 默认格式："Doc 1(Title: <title>) <text>\nDoc 2(...) ..."，
        # 再将其注入模板中的 {reference} 占位符，{question} 注入问题文本，
        # 最终返回完整 Prompt 字符串（str）或 chat messages 列表（List[Dict]，当 enable_chat=True 时）
        # 注：也可传 formatted_reference=str 跳过自动格式化，直接注入已拼好的字符串
        input_prompts = [                        # List[str | List[Dict]]: 每条问题对应一个 Prompt，长度 N
            self.prompt_template.get_string(question=q, retrieval_result=docs)  # str（或 List[Dict]，chat 模式）
            for q, docs in zip(questions, all_reranked)  # q: str 问题，docs: List[Dict] 精排文档
        ]

        # self.generator.generate(input_prompts) : 批量生成
        # 返回 List[str]，长度 N
        preds = self.generator.generate(input_prompts)  # List[str]: 批量生成结果，长度 N

        # ---- 写回 Dataset（来自官方 Pipeline 源码的惯用方式）----
        # dataset.update_output(field, values) : 向数据集添加新字段
        #   field  : str，字段名
        #   values : List，与数据集长度相同的列表
        dataset.update_output("pred", preds)                         # 写入预测回答字段
        dataset.update_output("retrieval_result", all_reranked)      # 写入精排检索结果字段

        # ---- 评估（调用父类方法）----
        # self.evaluate(dataset, do_eval, pred_process_fun) : 来自 BasicPipeline 源码
        dataset = self.evaluate(dataset, do_eval=do_eval, pred_process_fun=pred_process_fun)  # Dataset: 附加评估分数
        return dataset  # Dataset，含 pred 和 retrieval_result 字段


# ---- 使用自定义 Pipeline ----
# 创建包含 use_reranker=True 的 Config（CrossReranker 需要此开关）
reranker_config = Config(config_path, {"index_path": actual_index_path, "use_reranker": True})  # Config: 含 reranker 的配置
custom_reranker  = CrossReranker(reranker_config)     # CrossReranker: 官方类名（非 CrossEncoderReranker）
custom_pipeline  = RerankRAGPipeline(reranker_config, custom_reranker)  # RerankRAGPipeline: 自定义 Pipeline 实例

print("运行自定义 RerankRAG Pipeline...")  # 打印运行开始提示
custom_output = custom_pipeline.run(test_dataset, do_eval=False)  # Dataset: 含 pred/retrieval_result 字段

print("\n--- 自定义 Pipeline 结果 ---")  # 打印结果区标题
# 遍历输出数据集，item 含 question/pred/retrieval_result 等字段
for item in custom_output:  # item: Dict（question/pred/retrieval_result/golden_answers）
    print(f"\n问题   : {item['question']}")                   # 打印问题文本
    print(f"精排数 : {len(item['retrieval_result'])}")         # 打印精排后文档数
    print(f"预测   : {item['pred'][:80]}...")                  # 打印生成回答（前 80 字）

## 九、结果评估

FlashRAG Evaluator 的指标名称来自 `flashrag/evaluator/metrics.py` 源码，
每个指标类的 `metric_name` 属性决定配置中填写的名称，
`calculate_metric()` 返回的字典 key 决定 `evaluate()` 输出的实际 key。

| 配置中填写（`metrics` 列表） | `evaluate()` 输出的 key | 指标含义 |
|--------------------------|------------------------|----------|
| `"em"` | `em` | 预测与参考答案完全匹配（Exact Match） |
| `"f1"` | `f1` | 词级别 F1（精确率与召回率的调和平均） |
| `"acc"` | `acc` | 答案包含率（预测文本含参考答案则得 1） |
| `"precision"` | `precision` | 词级别精确率 |
| `"recall"` | `recall` | 词级别召回率 |
| `"retrieval_recall"` | `retrieval_recall_top{k}` | 检索召回率（Top-K 文档是否含答案） |
| `"retrieval_precision"` | `retrieval_precision_top{k}` | 检索精确率（Top-K 中含答案的比例） |
| `"rouge-1"` / `"rouge-2"` / `"rouge-l"` | `rouge-1` 等 | ROUGE 系列（需安装 `rouge` 库） |
| `"zh_rouge-1"` 等 | `zh_rouge-1` 等 | 中文 ROUGE（需 `rouge_chinese` + `jieba`） |
| `"bleu"` | `bleu` | BLEU 分数 |
| `"input_tokens"` | `avg_input_tokens` | 平均输入 token 数 |
| `"llm_judge"` | `llm_judge_score` | LLM 打分（0~1，需额外配置） |

In [ ]:
# flashrag.evaluator.Evaluator : 内置评估器（来自 flashrag/evaluator/evaluator.py）
# __init__ 时读取 config["metrics"] 列表，并为每个指标名实例化对应的 Metric 类
# 若指标名不在 BaseMetric 子类的 metric_name 中，会抛出 NotImplementedError
from flashrag.evaluator import Evaluator  # FlashRAG 内置评估器，支持 EM/F1/acc/retrieval_recall 等指标

# Evaluator(config) : 初始化评估器
evaluator = Evaluator(config_with_index)  # 从 config["metrics"] 中读取指标列表

print("=" * 55)              # 打印分隔线
print("各 Pipeline 评估结果对比")  # 打印标题
print("=" * 55)              # 打印分隔线

# pipeline_outputs : List[Tuple[str, Dataset]]，每项为 (Pipeline 名称, 输出 Dataset)
pipeline_outputs = [                           # List[Tuple[str, Dataset]]: 待评估的 Pipeline 及其输出
    ("Naive RAG",   naive_output),             # (str, Dataset): Naive RAG 名称和输出
    ("FLARE RAG",   flare_output),             # (str, Dataset): FLARE RAG 名称和输出
    ("Rerank RAG",  custom_output),            # (str, Dataset): 自定义 Rerank RAG 名称和输出
]

all_evals = {}  # Dict[str, Dict[str, float]]，汇总各 Pipeline 的评估结果
# 遍历 Pipeline 列表，依次评估每个 Pipeline 的输出
for name, output in pipeline_outputs:  # name: str Pipeline 名称，output: Dataset 输出数据集
    # evaluator.evaluate(dataset) : 计算评估指标
    # 参数:
    #   dataset : Dataset，必须含 pred（预测）和 golden_answers（参考答案）字段
    # 返回值:
    #   eval_result : Dict[str, float]，key=指标名，value=分数 [0,1]
    scores = evaluator.evaluate(output)  # Dict[str, float]: 指标名 → 分数 [0,1]
    all_evals[name] = scores             # 将当前 Pipeline 的分数存入汇总字典

# 收集所有出现过的指标名，排序后用于对齐表格列
all_metrics = sorted({m for s in all_evals.values() for m in s})  # List[str]
# header : str，表头字符串，Pipeline 名列宽 15，各指标列宽 10
header = f"{'Pipeline':<15}" + "".join(f"{m:>10}" for m in all_metrics)  # str: 表头字符串，各列宽 10
print(header)                     # 打印表头
print("-" * len(header))           # 打印表头分隔线
# 遍历各 Pipeline 的分数，打印对齐的表格行
for name, scores in all_evals.items():  # name: str，scores: Dict[str, float]
    row = f"{name:<15}"             # str: 行首 Pipeline 名，左对齐列宽 15
    for m in all_metrics:           # m: str 指标名
        # scores.get(m, 0.0) * 100 : float，转为百分比
        row += f"{scores.get(m, 0.0) * 100:>9.1f}%"  # str: 百分比格式（右对齐 9 字符）
    print(row)  # 打印当前 Pipeline 的评估结果行

## 十、完整端到端流程

将前九章所有组件串联为 6 个函数，清晰展示各步骤的拼接方式：

```
原始文档
   ↓ step1_build_corpus()       分块 → 写入 JSONL
   ↓ step2_build_index()        Index_Builder → FAISS 索引文件
   ↓ step3_setup_config()       创建 YAML → 加载 Config
   ↓ step4_prepare_dataset()    问答对 → Dataset 对象
   ↓ step5_run_pipeline()       batch_search → rerank → get_string → generate
   ↓ step6_evaluate()           Evaluator → 保存 JSON 报告
评估报告
```

### 10.1 端到端封装函数

In [ ]:
import os       # 操作系统接口，用于目录创建、路径拼接和环境变量读取
import glob     # 文件通配符匹配模块，用于查找构建完成的索引文件
import json     # JSON 序列化/反序列化模块，用于读写 JSONL 语料和报告
import yaml     # YAML 序列化/反序列化模块，用于写入配置文件
from typing import List, Dict, Optional  # 类型注解工具：List/Dict/Optional
from datetime import datetime            # 时间日期工具，用于生成报告时间戳

from langchain_text_splitters import RecursiveCharacterTextSplitter  # LangChain 分块工具

from flashrag.config              import Config             # FlashRAG 配置管理类
from flashrag.retriever           import DenseRetriever, CrossReranker  # 稠密检索器、CrossEncoder 重排序器
from flashrag.retriever.index_builder import Index_Builder  # FAISS / BM25 索引构建器
from flashrag.generator           import OpenaiGenerator    # OpenAI 兼容 API 生成器
from flashrag.pipeline            import BasicPipeline      # 所有 Pipeline 的基类
from flashrag.evaluator           import Evaluator          # 内置评估器
from flashrag.utils               import get_dataset        # 数据集加载工具函数
from flashrag.prompt              import PromptTemplate     # Prompt 模板管理类


def step1_build_corpus(                # 文档分块并写入 JSONL 语料文件
    raw_texts: List[Dict],             # List[Dict]: 原始文档列表，每项含 title 和 content
    corpus_path: str,                  # str: 输出的 JSONL 语料文件路径
    chunk_size: int = 200,             # int: 每个 chunk 的最大字符数，默认 200
    chunk_overlap: int = 30            # int: 相邻 chunk 的重叠字符数，默认 30
) -> int:                              # 返回值: int，写入的 chunk 总数
    """
    将原始文档列表用 LangChain RecursiveCharacterTextSplitter 分块，
    并写入 JSONL 语料文件供 FlashRAG 使用。

    注意：FlashRAG 本身没有内置分块工具，需借助外部库处理。

    Parameters
    ----------
    raw_texts    : List[Dict]，每个 Dict 含 title 和 content 字段
    corpus_path  : str，输出 JSONL 文件路径
    chunk_size   : int，每个 chunk 最大字符数
    chunk_overlap: int，相邻 chunk 重叠字符数

    Returns
    -------
    total_chunks : int，写入的 chunk 总数
    """
    os.makedirs(os.path.dirname(corpus_path), exist_ok=True)  # 递归创建语料目录，已存在不报错

    # RecursiveCharacterTextSplitter : 递归字符分块器
    # separators 按优先级从高到低：段落 → 换行 → 中文标点 → 空格 → 字符
    splitter = RecursiveCharacterTextSplitter(  # RecursiveCharacterTextSplitter: 分块器实例
        chunk_size=chunk_size,                       # int: 每块最大字符数
        chunk_overlap=chunk_overlap,                 # int: 相邻块重叠字符数
        separators=["\n\n", "\n", "。", "！", "？", "；", " ", ""],  # List[str]: 分隔符优先级列表
        length_function=len,                         # Callable: 字符计数函数
        is_separator_regex=False,                    # bool: 分隔符为普通字符串
    )

    total_chunks = 0  # int: chunk 计数器
    with open(corpus_path, "w", encoding="utf-8") as f:  # 文件句柄 f，覆盖写模式
        for doc_idx, doc in enumerate(raw_texts):  # doc_idx: int 文档序号，doc: Dict（title/content）
            # splitter.split_text(text) : 返回 List[str]，按分隔符优先级递归切分
            chunks: List[str] = splitter.split_text(doc["content"])  # List[str]: 切分后的文本块列表
            for chunk_idx, chunk in enumerate(chunks):  # chunk_idx: int 块序号，chunk: str 块文本
                if chunk.strip():  # 过滤空白块，避免写入无内容的记录
                    f.write(json.dumps({           # 将单条记录序列化为 JSON 行写入文件
                        "id":       f"{doc_idx}_{total_chunks}",  # str: 全局唯一 id
                        "title":    doc.get("title", ""),           # str: 文档标题
                        "contents": chunk,                          # str: chunk 正文
                    }, ensure_ascii=False) + "\n")  # str: 每条记录后附换行符，符合 JSONL 格式
                    total_chunks += 1  # 每成功写入一条 chunk，计数器加 1

    return total_chunks  # int: 总 chunk 数


def step2_build_index(                          # 构建 FAISS 向量检索索引文件
    corpus_path: str,                           # str: JSONL 语料文件路径
    index_dir: str,                             # str: 索引文件保存目录
    model_path: str = "BAAI/bge-small-zh-v1.5",  # str: 编码模型路径/Hub ID，默认小型中文模型
    retrieval_method: str = "bge",               # str: 检索方式名称，决定编码器类型
) -> str:                                       # 返回值: str，生成的 FAISS 索引文件路径
    """
    使用 Index_Builder 构建 FAISS 索引文件。

    Parameters
    ----------
    corpus_path      : str，JSONL 语料文件路径
    index_dir        : str，索引保存目录
    model_path       : str，编码模型路径/Hub ID
    retrieval_method : str，检索方式名称（决定编码器类型）

    Returns
    -------
    index_path : str，生成的 FAISS 索引文件路径
    """
    # Index_Builder 参数（来自源码 __init__）
    builder = Index_Builder(           # Index_Builder: 初始化索引构建器
        retrieval_method = retrieval_method,  # str: 检索方式
        model_path       = model_path,         # str: 编码模型路径
        corpus_path      = corpus_path,        # str: JSONL 语料
        save_dir         = index_dir,          # str: 索引保存目录
        max_length       = 128,                # int: 编码最大长度
        batch_size       = 256,                # int: 编码批次大小
        use_fp16         = True,               # bool: 使用 fp16
        faiss_type       = "Flat",             # str: 精确检索类型
    )
    builder.build_index()  # 执行构建，在 save_dir 中生成 .index 文件

    # 查找生成的索引文件
    # 来自源码：文件名格式 = {retrieval_method}_{faiss_type}.index（如 bge_Flat.index）
    index_files = glob.glob(f"{index_dir}/*.index")  # List[str]: 匹配 .index 文件（扩展名是 .index 不是 .faiss）
    assert index_files, f"索引构建失败，{index_dir} 中未找到 .index 文件"  # 断言索引文件存在，否则抛出 AssertionError
    return index_files[0]  # str: 第一个索引文件路径


def step3_setup_config(                           # 创建 YAML 配置文件并返回 Config 对象
    workspace: str,                               # str: 根工作目录路径
    dataset_name: str,                            # str: 数据集名称（作为 data_dir 的子目录）
    corpus_path: str,                             # str: JSONL 语料文件路径
    index_path: str,                              # str: 已构建的 FAISS 索引文件路径
    api_key: str,                                 # str: 生成器 API 密钥
    base_url: str,                                # str: 生成器 API 根地址
    generator_model: str = "deepseek-chat",       # str: 生成模型 ID，默认 deepseek-chat
    retrieval_model: str = "BAAI/bge-small-zh-v1.5",  # str: 编码模型路径，默认小型中文模型
    rerank_model: str = "BAAI/bge-reranker-base",  # str: reranker 模型路径
    retrieval_topk: int = 5,                      # int: 检索返回 Top-K 文档数，默认 5
    rerank_topk: int = 3,                         # int: 重排序保留文档数，默认 3
) -> Config:                                      # 返回值: Config，加载完毕的 FlashRAG 配置对象
    """
    创建 YAML 配置文件并返回加载后的 Config 对象。

    Returns
    -------
    config : Config，加载完毕的 FlashRAG 配置对象
    """
    os.makedirs(workspace, exist_ok=True)  # 创建工作目录，已存在不报错

    cfg = {                              # Dict: FlashRAG 配置字典，仅覆盖与默认值不同的字段
        "save_dir":             f"{workspace}/output",       # str: 评估结果保存目录
        "gpu_id":               "0",                          # str: GPU 编号
        "seed":                 2024,                          # int: 随机种子
        "save_note":            "e2e",                         # str: 保存目录后缀
        "corpus_path":          corpus_path,                 # str: JSONL 语料
        "index_path":           index_path,                  # str: FAISS 索引文件（已存在）
        "retrieval_method":     "bge",                         # str: 检索方式名称
        "retrieval_model_path": retrieval_model,              # str: 编码模型路径
        "retrieval_topk":       retrieval_topk,               # int: 检索返回 Top-K 文档数
        "retrieval_batch_size": 256,                           # int: 编码批次大小
        "bm25_backend":         "bm25s",                     # str: 有效值 "bm25s" / "pyserini"
        "use_reranker":         True,                        # bool: 启用重排序
        "rerank_model_name":    "bge-reranker-base",          # str: reranker 模型别名
        "rerank_model_path":    rerank_model,                 # str: reranker 模型路径
        "rerank_topk":          rerank_topk,                  # int: 重排序后保留文档数
        "rerank_max_length":    512,                           # int: reranker 最大 token 数
        "rerank_batch_size":    64,                            # int: reranker 批次大小
        "framework":            "openai",                      # str: 生成器后端类型
        "generator_model":      generator_model,               # str: 生成模型 ID
        "generator_max_input_len": 4096,                       # int: Prompt 最大 token 数
        "generator_batch_size": 1,                             # int: 并发生成请求数
        "generation_params":    {"max_tokens": 512, "temperature": 0.1},  # Dict: 生成超参数
        "openai_setting":       {"api_key": api_key, "base_url": base_url},  # Dict: OpenAI SDK 初始化参数
        # dataset_path = data_dir / dataset_name（Config 自动计算）
        "data_dir":             workspace,                   # str: 数据集根目录
        "dataset_name":         dataset_name,                # str: 数据集名（子目录）
        "split":                ["test"],                     # List[str]: 使用的数据集分片
        "test_sample_num":      None,                         # int|None: 评估采样数，None 表示全量
    }  # end of cfg

    config_path = f"{workspace}/config.yaml"  # str: 配置文件路径
    os.makedirs(workspace, exist_ok=True)      # 再次确保工作目录存在（防止 workspace 刚被传入）
    with open(config_path, "w", encoding="utf-8") as f:  # 文件句柄 f，写入 YAML 配置文件
        yaml.dump(cfg, f, allow_unicode=True, default_flow_style=False, sort_keys=False)  # 块式 YAML，保留中文

    return Config(config_path, {})  # Config 对象：从写入的 YAML 文件加载完整配置


def step4_prepare_dataset(              # 将问答对写入 JSONL 并加载为 FlashRAG Dataset
    qa_pairs: List[Dict],               # List[Dict]: 问答对列表，每项含 id/question/golden_answers
    workspace: str,                     # str: 工作目录路径（data_dir）
    dataset_name: str,                  # str: 数据集名称（子目录名）
    config: Config                      # Config: 配置对象（用于 get_dataset 读取 dataset_path）
):                                      # 返回值: Dataset，FlashRAG 格式测试集
    """
    将问答对写入正确路径的 JSONL 文件，并加载为 FlashRAG Dataset。

    路径规则（来自 Config 源码）：
      dataset_path = data_dir / dataset_name
      文件须放在 dataset_path/test.jsonl

    Returns
    -------
    dataset : Dataset，FlashRAG 格式数据集
    """
    # 数据集目录 = workspace / dataset_name
    data_subdir = os.path.join(workspace, dataset_name)  # str: 数据集子目录路径
    os.makedirs(data_subdir, exist_ok=True)               # 创建数据集子目录，已存在不报错

    jsonl_path = os.path.join(data_subdir, "test.jsonl")  # str: JSONL 文件路径（必须命名为 test.jsonl）
    with open(jsonl_path, "w", encoding="utf-8") as f:   # 文件句柄 f，写入问答对
        for item in qa_pairs:                              # item: Dict（id/question/golden_answers）
            f.write(json.dumps(item, ensure_ascii=False) + "\n")  # 每条问答对序列化为一行 JSON

    return get_dataset(config)["test"]  # Dataset 对象：test 分片，含 question/golden_answers 字段


def step5_run_pipeline(config: Config, dataset, reranker: CrossReranker):  # 执行检索→重排序→生成完整流程
    """
    执行「检索 → 重排序 → 生成」完整流程。

    使用正确的 API：
    - retriever.batch_search() （非 retrieve()）
    - reranker.rerank() 返回 (docs, scores) 元组
    - prompt_template.get_string() （非 build_prompt()）
    - generator.generate()

    Returns
    -------
    dataset : Dataset，增加了 pred 和 retrieval_result 字段
    """
    retriever = DenseRetriever(config)   # DenseRetriever: 加载已有 FAISS 索引
    prompt_template = PromptTemplate(config)  # PromptTemplate: 使用默认模板
    generator_obj = OpenaiGenerator(config)   # OpenaiGenerator: API 生成器

    questions = dataset.question  # List[str]: 数据集所有问题，长度 N

    print("  [1/3] 批量粗排检索...")  # 打印第 1 步进度提示
    # batch_search 返回 List[List[Dict]]，shape: (N, retrieval_topk)
    all_candidates = retriever.batch_search(questions)  # List[List[Dict]]: 粗排候选文档

    print("  [2/3] 精排重排序...")  # 打印第 2 步进度提示
    # rerank 返回 (List[List[Dict]], List[List[float]])，两个长度 N 的列表
    all_reranked, _ = reranker.rerank(questions, all_candidates)  # List[List[Dict]]: 精排后文档，_ 为分数（忽略）

    print("  [3/3] 构造 Prompt 并生成...")  # 打印第 3 步进度提示
    # get_string 返回 str 或 List[Dict]（取决于是否设置了 system_prompt）
    input_prompts = [                         # List[str | List[Dict]]: 每条问题的 Prompt，长度 N
        prompt_template.get_string(question=q, retrieval_result=docs)  # str 或 List[Dict]
        for q, docs in zip(questions, all_reranked)  # q: str 问题，docs: List[Dict] 精排文档
    ]

    preds = generator_obj.generate(input_prompts)  # List[str]: 批量生成结果，长度 N

    dataset.update_output("pred", preds)                     # 写入预测回答字段
    dataset.update_output("retrieval_result", all_reranked)  # 写入精排检索结果字段
    return dataset  # Dataset，含 pred 和 retrieval_result 字段


def step6_evaluate(config: Config, output_dataset, extra_info: Optional[Dict] = None) -> Dict:  # 评估并保存 JSON 报告
    """
    评估 Pipeline 输出并保存 JSON 报告。

    Returns
    -------
    report : Dict，含所有评估指标分数的报告字典
    """
    # Evaluator.evaluate 返回 Dict[str, float]，key=指标名，value∈[0,1]
    scores = Evaluator(config).evaluate(output_dataset)  # Dict[str, float]: 各指标原始分数（[0,1] 区间）

    # 注意（来自 config.py 源码）：
    #   Config.__getitem__ 使用 final_config.get(item)，缺失 key 返回 None，不抛 KeyError
    #   Config 没有 .get() 方法（__getattr__ 对非配置 key 抛 AttributeError）
    #   因此不能用 config.get("key", default)，要用 config["key"] or default
    report = {                                                 # Dict: 完整评估报告字典
        "timestamp":       datetime.now().strftime("%Y%m%d_%H%M%S"),  # str: 时间戳（格式 YYYYMMDDHHmmss）
        "retrieval_model": config["retrieval_model_path"] or "N/A",   # str: 检索编码模型路径
        "rerank_model":    config["rerank_model_path"] or "N/A",    # str: reranker 模型路径，or 处理 None
        "generator_model": config["generator_model"] or "N/A",        # str: 生成模型 ID
        "retrieval_topk":  config["retrieval_topk"],                   # int: 检索 Top-K
        "rerank_topk":     config["rerank_topk"] or "N/A",          # int|str: 重排序 Top-K，or 处理 None
        # 将 [0,1] 的分数转为百分比
        "scores":          {k: round(v * 100, 2) for k, v in scores.items()},  # Dict[str, float]: 百分比分数
    }
    if extra_info:              # 若传入了额外信息字典，则合并到报告中
        report.update(extra_info)   # 将 extra_info 的所有键值对更新到 report

    save_dir = config["save_dir"]               # str: 评估结果保存目录
    os.makedirs(save_dir, exist_ok=True)         # 创建保存目录，已存在不报错
    save_path = os.path.join(save_dir, f"report_{report['timestamp']}.json")  # str: 报告文件路径
    with open(save_path, "w", encoding="utf-8") as f:  # 文件句柄 f，写入 JSON 报告
        json.dump(report, f, ensure_ascii=False, indent=2)  # 缩进 2 格，保留中文字符

    print(f"  报告已保存: {save_path}")   # 打印报告文件保存路径
    return report  # Dict: 完整评估报告


print("端到端函数定义完毕")   # 打印所有函数定义加载成功的确认信息

### 10.2 一键运行完整流程

In [ ]:
# ============================================================
# 用户配置区（仅需修改此处）
# ============================================================
E2E_WORKSPACE    = "flashrag_e2e"                        # str: 工作目录
E2E_DATASET_NAME = "mydata"                               # str: 数据集名
E2E_API_KEY      = os.environ.get("DEEPSEEK_API_KEY", "your-key")  # str: API 密钥
E2E_BASE_URL     = "https://api.deepseek.com/v1"         # str: API 地址
E2E_EMBED_MODEL  = "BAAI/bge-small-zh-v1.5"             # str: 向量编码模型
E2E_RERANK_MODEL = "BAAI/bge-reranker-base"              # str: 重排序模型
E2E_GEN_MODEL    = "deepseek-chat"                       # str: 生成模型

# ---- 原始文档（替换为你自己的文档）----
e2e_raw_texts = [                         # List[Dict]: 端到端演示原始文档，每项含 title 和 content
    {"title": "RAG 概述",                 # str: 文档 1 标题
     "content": "RAG 即检索增强生成，结合检索和大模型，解决幻觉和知识更新问题。"   # str: 文档 1 正文
                "三阶段：朴素 RAG（检索生成）、高级 RAG（重排序/改写）、模块化 RAG。"},
    {"title": "FlashRAG",                # str: 文档 2 标题
     "content": "FlashRAG 是中国人民大学开源 RAG 框架，内置 Naive RAG、Self-RAG、FLARE 等算法，"  # str: 文档 2 正文
                "支持 OpenaiGenerator、HFCausalLMGenerator，以及 DenseRetriever 和 BM25Retriever。"},
    {"title": "高级 RAG",                # str: 文档 3 标题
     "content": "FLARE 在生成低置信度 token 时主动检索，将检索和生成交织进行。"     # str: 文档 3 正文
                "Self-RAG 通过反思 token 自主决定是否检索。"},
]

# ---- 问答测试集（替换为你自己的问题）----
e2e_qa = [                                # List[Dict]: 端到端测试问答数据集
    {"id": "1", "question": "RAG 的三个发展阶段是什么？",          # str: 问题 1
     "golden_answers": ["朴素 RAG、高级 RAG、模块化 RAG"]},        # List[str]: 参考答案列表
    {"id": "2", "question": "FlashRAG 支持哪些生成器后端？",        # str: 问题 2
     "golden_answers": ["OpenaiGenerator、HFCausalLMGenerator"]},  # List[str]: 参考答案列表
    {"id": "3", "question": "FLARE 算法的核心特点？",               # str: 问题 3
     "golden_answers": ["低置信度 token 时主动检索"]},              # List[str]: 参考答案列表
]

# ============================================================
# 完整端到端流程（6 步）
# ============================================================
e2e_corpus_path = f"{E2E_WORKSPACE}/corpus/corpus.jsonl"   # str: 端到端语料文件路径
e2e_index_dir   = f"{E2E_WORKSPACE}/index"                 # str: 端到端索引保存目录

print("[Step 1/6] 文档分块 → 写入 JSONL")   # 打印第 1 步提示
n = step1_build_corpus(e2e_raw_texts, e2e_corpus_path)   # int: 写入的 chunk 总数
print(f"   → {n} 个 chunk")                 # 打印分块统计结果

print("\n[Step 2/6] 构建 FAISS 索引（Index_Builder）")  # 打印第 2 步提示
e2e_index_path = step2_build_index(          # str: 构建完成的索引文件路径
    e2e_corpus_path, e2e_index_dir,          # str: 语料路径和索引保存目录
    model_path=E2E_EMBED_MODEL, retrieval_method="bge"   # str: 编码模型和检索方式
)
print(f"   → 索引文件: {e2e_index_path}")   # 打印索引文件路径

print("\n[Step 3/6] 创建 Config")           # 打印第 3 步提示
e2e_config = step3_setup_config(             # Config: 端到端完整配置对象
    workspace       = E2E_WORKSPACE,         # str: 工作目录
    dataset_name    = E2E_DATASET_NAME,      # str: 数据集名称
    corpus_path     = e2e_corpus_path,       # str: 语料文件路径
    index_path      = e2e_index_path,        # str: 索引文件路径
    api_key         = E2E_API_KEY,           # str: API 密钥
    base_url        = E2E_BASE_URL,          # str: API 根地址
    generator_model = E2E_GEN_MODEL,         # str: 生成模型 ID
    retrieval_model = E2E_EMBED_MODEL,       # str: 编码模型路径
    rerank_model    = E2E_RERANK_MODEL,      # str: reranker 模型路径
)
print(f"   → dataset_path={e2e_config['dataset_path']}")  # 打印数据集实际路径

print("\n[Step 4/6] 准备数据集")            # 打印第 4 步提示
e2e_dataset = step4_prepare_dataset(e2e_qa, E2E_WORKSPACE, E2E_DATASET_NAME, e2e_config)  # Dataset: 测试集
print(f"   → {len(e2e_dataset)} 条问答")   # 打印数据集条数

print("\n[Step 5/6] 运行 Pipeline（检索 → 重排序 → 生成）")  # 打印第 5 步提示
# 初始化重排序器（CrossReranker，来自官方源码）
e2e_reranker = CrossReranker(e2e_config)   # CrossReranker: 官方类名
e2e_output   = step5_run_pipeline(e2e_config, e2e_dataset, e2e_reranker)  # Dataset: 含 pred 字段

print("\n[Step 6/6] 评估并保存报告")         # 打印第 6 步提示
e2e_report = step6_evaluate(                # Dict: 完整评估报告字典
    e2e_config, e2e_output,                 # Config: 配置对象，Dataset: 含 pred 字段的输出
    extra_info={"experiment": "e2e_rerank_rag"}   # Dict: 附加实验标识信息
)

# ---- 打印汇总 ----
print("\n" + "=" * 55)  # 打印分隔线
print("端到端流程完成！评估结果:")  # 打印标题
# 遍历报告中的指标和分数，metric 为 str，score 为 float（已转为百分比）
for metric, score in e2e_report["scores"].items():  # metric: str 指标名，score: float 百分比分数
    print(f"  {metric:<12}: {score:.2f}%")  # 打印指标名（左对齐 12 字符）和百分比分数

print("\n问题预测对照:")  # 打印对照区标题
# 遍历输出数据集，每条 item 含 question/pred/golden_answers
for item in e2e_output:    # item: Dict（question/pred/golden_answers/retrieval_result）
    print(f"  Q: {item['question']}")           # 打印问题文本
    print(f"  A: {item['pred'][:80]}...")        # 打印预测回答（前 80 字）
    print(f"  ✓: {item['golden_answers']}")      # 打印参考答案列表
    print()  # 打印空行，分隔不同问答对

### 10.3 组件替换速查表

所有组件可独立替换，其余代码无需修改：

| 替换目标 | 修改位置 | 示例 |
|----------|---------|------|
| 换检索模型 | `step3_setup_config(retrieval_model=...)` | `"BAAI/bge-large-zh-v1.5"` |
| 换重排序模型 | `step3_setup_config(rerank_model=...)` | `"cross-encoder/ms-marco-MiniLM-L-6-v2"` |
| 换生成模型 | `step3_setup_config(generator_model=...)` | `"gpt-4o-mini"` |
| 换本地生成 | `step3_setup_config(framework="hf")` | 需配合 `generator_model_path` |
| 换 BM25 检索 | `retrieval_method="bm25"` | `bm25_backend="bm25s"` |
| 关闭重排序 | `use_reranker=False` | 删除 `step5` 中的 `rerank` 调用 |
| 换 Pipeline 算法 | 替换 `step5` 的 `BasicPipeline` 子类 | `FLAREPipeline`、`SelfRAGPipeline` |

## 十一、总结

### 本 Notebook 知识点回顾

| 章节 | 核心内容 | 关键 API（均经官方源码验证） |
|------|---------|----------------------------|
| 3.1 | YAML 配置 | `Config(path, override_dict)` |
| 4.1 | 文档分块 | `RecursiveCharacterTextSplitter` (LangChain) → JSONL |
| 4.2 | 构建索引 | `Index_Builder(...).build_index()` |
| 4.3 | 加载检索器 | `DenseRetriever(config)`（索引须预先构建） |
| 5.1 | 稠密检索 | `retriever.batch_search(queries)` |
| 5.2 | BM25 检索 | `BM25Retriever(config).batch_search(queries)` |
| 5.3 | 混合检索 | `use_multi_retriever=True` → `MultiRetrieverRouter` |
| 6 | 重排序 | `CrossReranker(config).rerank(queries, docs_list)` → `(docs, scores)` |
| 7.1 | API 生成 | `OpenaiGenerator(config).generate(input_list)` |
| 7.2 | 本地生成 | `get_generator(config)` → `HFCausalLMGenerator` |
| 7.3 | Prompt 模板 | `PromptTemplate(config).get_string(question=q, retrieval_result=r)` |
| 8.2 | Naive RAG | `SequentialPipeline(config).run(dataset)` |
| 8.3 | FLARE | `FLAREPipeline(config).run(dataset)`（需开启 logprobs） |
| 8.4 | HyDE（手动） | `generate(hypo_prompt)` → `batch_search(hypo_doc)` |
| 8.5 | 自定义 Pipeline | 继承 `BasicPipeline`，重写 `run()` |
| 9 | 评估 | `Evaluator(config).evaluate(dataset)` |
| **10** | **端到端** | **step1→step6 串联，一键跑通** |
### RAG 系统建设核心清单

```
□ 1. 语料准备：PDF/HTML → 纯文本 → RecursiveCharacterTextSplitter (LangChain) → JSONL
□ 2. 索引构建：Index_Builder(...).build_index()
     Dense: 生成 {save_dir}/{method}_{faiss_type}.index
     BM25:  生成 {save_dir}/bm25/ 目录（Dense 和 BM25 都需要预构建！）
□ 3. 配置管理：data_dir+dataset_name → Config（data_dir/dataset_name/test.jsonl）
□ 4. 检索策略：Dense（语义）/ BM25（关键词）/ 多路混合（两者兼顾）
□ 5. 重排序  ：CrossReranker.rerank() 返回 (docs, scores) 元组
□ 6. 生成后端：API（OpenaiGenerator）/ 本地（get_generator() → HFCausalLMGenerator）
□ 7. Pipeline ：SequentialPipeline / FLAREPipeline / 自定义继承 BasicPipeline
□ 8. 评估指标：Evaluator（EM/F1/acc，来自 basic_config.yaml 默认 metrics）
```